# S4 — CMIP6 spatial relationship classification: P→Q, Surface runoff→Q, Latent heat→Q

Active workflow: compute **MIC and dCor association gates**, then classify **branch structure** on panels that pass the gate (Sections 14–15), and inspect the same panels as Hexbin density (Section 16).

Earlier pair-overlay / LOWESS-shape / heatmap cells are retained but disabled.

**中文说明：** 当前主流程是先算 MIC 和 dCor 两个关联 gate，通过后再做第 14–15 节的 branch 分类，最后用第 16 节的 Hexbin 5×6 密度图对照。前面的散点总图、LOWESS 形态和热力图先注释掉，不参与这次运行。


## 1. Configuration / 配置

Locate S3 zone climatology output and explicitly load the case-local classifier module. MIC / dCor thresholds come from `classifier.py` defaults.

**中文说明：** 定位S3分区气候态输出，并显式加载 `classifier.py`。后面的 MIC / dCor gate 阈值直接用这个文件里的 defaults。


In [ ]:
from __future__ import annotations

import importlib
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.lines import Line2D

warnings.filterwarnings('ignore', category=FutureWarning)


def locate_case_dir() -> Path:
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for candidate in candidates:
        if (candidate / 'cmip_utils.py').exists() and candidate.name == 'caseA':
            return candidate
        nested = candidate / 'case' / 'caseA'
        if (nested / 'cmip_utils.py').exists():
            return nested
    raise FileNotFoundError('Could not locate case/caseA/cmip_utils.py')


CASE_DIR = locate_case_dir()
DATA_ROOT = Path('/Volumes/mimi-T9/CMIP6')

CLASSIFIER_PATH = CASE_DIR / 'classifier0825-branch.py'
if not CLASSIFIER_PATH.exists():
    raise FileNotFoundError(f'Case-local classifier not found: {CLASSIFIER_PATH}')
classifier_spec = importlib.util.spec_from_file_location(
    'caseA_classifier', CLASSIFIER_PATH
)
if classifier_spec is None or classifier_spec.loader is None:
    raise ImportError(f'Cannot create import spec for {CLASSIFIER_PATH}')
classifier_module = importlib.util.module_from_spec(classifier_spec)
classifier_spec.loader.exec_module(classifier_module)
classify_relationship = classifier_module.classify_relationship
CLASSIFIER_DEFAULTS = classifier_module.DEFAULTS

MODELS = [
    'CESM2', 'CNRM-CM6-1', 'CanESM5', 'GFDL-CM4', 'CMCC-CM2-SR5',
]
EXPERIMENTS = ['historical']
START_YEAR = 1985
END_YEAR = 2014
WRITE_OUTPUTS = True
TRIM_LOWER_QUANTILE = 0.01
TRIM_UPPER_QUANTILE = 0.99

VARIABLE_PAIRS = [
    ('P',          'Q', 'P → Q'),
    ('ET',         'Q', 'ET → Q'),
    ('hfls',       'Q', 'hfls → Q'),
    ('hfss',       'Q', 'hfss → Q'),
    ('tran',       'Q', 'tran → Q'),
    ('evspsblsoi', 'Q', 'evspsblsoi → Q'),
    ('P', 'ET', 'P → ET'),
    ('mrros',      'Q', 'mrros → Q'),
    ('mrso',       'Q', 'mrso → Q'),
    ('mrsos',      'Q', 'mrsos → Q'),
    ('lai',        'Q', 'lai → Q'),
    ('tas',        'Q', 'tas → Q'),
    ('prsn',       'Q', 'prsn → Q'),
    ('rlds',       'Q', 'rlds → Q'),
    ('rlus',       'Q', 'rlus → Q'),
    ('rsds',       'Q', 'rsds → Q'),
    ('rsus',       'Q', 'rsus → Q'),
]

DOMAINS = [
    ('all_land', 'All land', None),
    ('WW', 'Wet–warm', lambda t: t['analysis_zone'] == 'WW'),
    ('WD', 'Dry–warm', lambda t: t['analysis_zone'] == 'WD'),
    ('CW', 'Wet–cold', lambda t: t['analysis_zone'] == 'CW'),
    ('CD', 'Dry–cold', lambda t: t['analysis_zone'] == 'CD'),
    ('LI', 'Land ice', lambda t: t['analysis_zone'] == 'LI'),
]

# Colours sampled from the climate-region reference map.
ZONE_COLORS = {
    'all_land': '#6B7280',
    'non_ice_land': '#6B7280',
    'WW': '#008070',  # wet-warm, dark teal
    'WD': '#A06018',  # dry-warm, rust brown
    'CW': '#80C8C0',  # wet-cold, mint
    'CD': '#D8C078',  # dry-cold, beige
    'LI': '#3B82F6',  # land ice, blue points
}
ZONE_LEGEND = [
    ('WW', 'Wet–warm'),
    ('CW', 'Wet–cold'),
    ('CD', 'Dry–cold'),
    ('WD', 'Dry–warm'),
    ('LI', 'Land ice'),
]

# Panel background by classification family.
# Simple path = warm tints; complex path = cool tints.
CLASS_FACECOLORS = {
    # Simple path: warm hues
    'Linear': '#FFE8D2',
    'Saturation': '#F6D7C8',
    'Acceleration': '#F7E7A8',
    # Branch path: cool blue
    'Branch': '#D7E8F6',
    'Candidate branch': '#E0D8F2',
    # Complex path: teal / grey
    'Complex': '#CDEEEE',
    'Uncertain': '#E8E8E8',
    'No Global Relationship': '#FFFFFF',
    'No global relationship': '#FFFFFF',
    'No result': '#FFFFFF',
    'Insufficient data': '#F4F4F4',
}

# One stable colour per model on the multi-model overlay scatters.
MODEL_COLOR_CYCLE = [
    '#0072B2', '#D55E00', '#009E73', '#CC79A7', '#E69F00',
    '#56B4E9', '#882255', '#44AA99', '#332288', '#117733',
    '#AA4499', '#88CCEE', '#999933', '#DDCC77', '#000000',
]

plt.rcParams.update({
    'figure.dpi': 120, 'savefig.dpi': 200,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'font.family': 'DejaVu Sans', 'axes.edgecolor': '#30343B',
    'axes.linewidth': 1.0, 'axes.titlesize': 14,
    'axes.titleweight': 'semibold', 'axes.labelsize': 13,
    'xtick.labelsize': 11, 'ytick.labelsize': 11,
    'legend.fontsize': 11, 'figure.titlesize': 17,
})

S4_OUTPUT_DIR = CASE_DIR / 'output' / 'S4'
S4_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR = S4_OUTPUT_DIR / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
PAIR_SLUGS = {label: f'{x}_{y}' for x, y, label in VARIABLE_PAIRS}
SHOW_LOWESS = False  # 是否在 scatter grid 上绘制 LOWESS 曲线

print(f'Case directory: {CASE_DIR}')
print(f'Classifier path:     {CLASSIFIER_PATH}')
print(f'Classifier defaults: {CLASSIFIER_DEFAULTS}')
print(f'Domains: {[d[0] for d in DOMAINS]}')
print(f'Variable pairs: {[v[2] for v in VARIABLE_PAIRS]}')


## 2. Load S3 zone climatology / 加载S3分区气候态

Read the zone climatology table produced by S3. Verify expected columns and zone labels.

**中文说明：** 读取S3输出的分区气候态表，验证列名和分区标签。

In [ ]:
RUNS = []
SKIPPED_MODELS = []
for model in MODELS:
    for experiment in EXPERIMENTS:
        search_root = DATA_ROOT / model / experiment
        zone_name = f'zone_climatology_{START_YEAR}_{END_YEAR}.parquet'
        zone_paths = sorted(search_root.glob(f'*/*/land/zones/{zone_name}'))
        if not zone_paths:
            SKIPPED_MODELS.append(model)
            print(f'{model}: no S3 zone climatology, skip')
            continue
        for zp in zone_paths:
            zones_dir = zp.parent
            run_root = zones_dir.parent.parent
            member = run_root.parent.name
            grid = run_root.name
            RUNS.append({
                'model': model,
                'experiment': experiment,
                'member': member,
                'grid': grid,
                'zone_climatology_path': zp,
                'zones_dir': zones_dir,
                'run_root': run_root,
            })

for run in RUNS:
    t = pd.read_parquet(run['zone_climatology_path'])
    t.rename(columns={'R': 'Q'}, inplace=True)
    run['clim'] = t
    required = {'P', 'ET', 'Q', 'analysis_zone', 'surface_class', 'core_zone_cell', 'grid_id', 'land_area'}
    missing = required - set(t.columns)
    if missing:
        raise KeyError(f'Missing columns in S3 output: {sorted(missing)}')
    print(f"{run['model']} {run['member']} {run['grid']}: {len(t)} rows")
    print(f"  analysis_zone: {sorted(t['analysis_zone'].unique())}")
    print(f"  surface_class: {sorted(t['surface_class'].unique())}")

print(f'\nLoaded {len(RUNS)} runs, skipped {len(SKIPPED_MODELS)}: {SKIPPED_MODELS}')
if not RUNS:
    raise FileNotFoundError('No S3 zone climatology found for any requested model')


## 3. Association gates — MIC and dCor / MIC 与 dCor 关联门槛

Compute MIC and dCor for every model × domain × pair × raw/trimmed panel. Each metric is labelled `strong_global` / `intermediate_global` / `no_global` using the classifier defaults (strong 0.8, intermediate 0.4). The old power-law / SiZer / Y-dip tree is not run here.

**中文说明：** 这里只算 MIC 和 dCor 两个 gate，不再跑原来的形状分类树。两个分数都会保存；第 14–15 节用开关决定哪个（或两个一起）打开 branch 检测。


In [ ]:
import os
import time as _time
from concurrent.futures import ThreadPoolExecutor, as_completed

# Reload classifier.py from disk so edits apply without restarting the kernel.
classifier_spec = importlib.util.spec_from_file_location(
    'caseA_classifier', CLASSIFIER_PATH
)
classifier_module = importlib.util.module_from_spec(classifier_spec)
classifier_spec.loader.exec_module(classifier_module)
compute_mic = classifier_module.compute_mic
compute_dcor = classifier_module.compute_dcor
CLASSIFIER_DEFAULTS = classifier_module.DEFAULTS
print(f'Reloaded classifier: {CLASSIFIER_PATH}')
print(f'Defaults: {CLASSIFIER_DEFAULTS}')

MIC_STRONG = CLASSIFIER_DEFAULTS['mic_strong']
MIC_INTERMEDIATE = CLASSIFIER_DEFAULTS['mic_intermediate']
DCOR_STRONG = CLASSIFIER_DEFAULTS['dcor_strong']
DCOR_INTERMEDIATE = CLASSIFIER_DEFAULTS['dcor_intermediate']


def association_gate(score, strong, intermediate):
    """Label one association score against the strong / intermediate thresholds."""
    if score is None or not np.isfinite(score):
        return 'no_global'
    if score >= strong:
        return 'strong_global'
    if score >= intermediate:
        return 'intermediate_global'
    return 'no_global'


def prepare_pair_versions(x, y, groups=None):
    """Return finite raw values and a paired 1–99% trimmed sensitivity sample."""
    x_arr = np.asarray(x, dtype=float)
    y_arr = np.asarray(y, dtype=float)
    finite = np.isfinite(x_arr) & np.isfinite(y_arr)
    x_raw = x_arr[finite]
    y_raw = y_arr[finite]
    groups_raw = None if groups is None else np.asarray(groups)[finite]
    if len(x_raw) == 0:
        bounds = (np.nan, np.nan, np.nan, np.nan)
        keep = np.zeros(0, dtype=bool)
    else:
        x_lo, x_hi = np.quantile(
            x_raw, [TRIM_LOWER_QUANTILE, TRIM_UPPER_QUANTILE]
        )
        y_lo, y_hi = np.quantile(
            y_raw, [TRIM_LOWER_QUANTILE, TRIM_UPPER_QUANTILE]
        )
        bounds = (float(x_lo), float(x_hi), float(y_lo), float(y_hi))
        keep = (
            (x_raw >= x_lo) & (x_raw <= x_hi)
            & (y_raw >= y_lo) & (y_raw <= y_hi)
        )
    return {
        'raw': (x_raw, y_raw),
        'trimmed': (x_raw[keep], y_raw[keep]),
        'raw_groups': groups_raw,
        'trimmed_groups': None if groups_raw is None else groups_raw[keep],
        'trim_keep': keep,
        'bounds': bounds,
    }


def run_association_gate(x_clean, y_clean, x_name, y_name, domain_name,
                         sample_name, outlier_version, raw_n_cells, bounds):
    """Compute MIC and dCor gates only; no shape / old-tree classification."""
    x_clean = np.asarray(x_clean, dtype=float)
    y_clean = np.asarray(y_clean, dtype=float)
    n = len(x_clean)
    x_lo, x_hi, y_lo, y_hi = bounds
    n_removed = int(raw_n_cells - n)
    mic_value = np.nan
    dcor_value = np.nan
    if n >= 10 and np.ptp(x_clean) > 0 and np.ptp(y_clean) > 0:
        mic_value = float(compute_mic(x_clean, y_clean))
        dcor_value = float(compute_dcor(x_clean, y_clean))
    mic_gate = association_gate(mic_value, MIC_STRONG, MIC_INTERMEDIATE)
    dcor_gate = association_gate(dcor_value, DCOR_STRONG, DCOR_INTERMEDIATE)
    return {
        'domain': domain_name,
        'sample': sample_name,
        'outlier_version': outlier_version,
        'x_var': x_name,
        'y_var': y_name,
        'pair': f'{x_name} → {y_name}',
        'n_cells': n,
        'raw_n_cells': int(raw_n_cells),
        'n_removed': n_removed,
        'removed_fraction': (n_removed / raw_n_cells if raw_n_cells else np.nan),
        'x_trim_lower': x_lo,
        'x_trim_upper': x_hi,
        'y_trim_lower': y_lo,
        'y_trim_upper': y_hi,
        'mic': mic_value,
        'dcor': dcor_value,
        'mic_gate': mic_gate,
        'dcor_gate': dcor_gate,
        'mic_pass': mic_gate != 'no_global',
        'dcor_pass': dcor_gate != 'no_global',
    }


# ── Pre-build all jobs as numpy arrays ──────────────────────
# Trimming is computed GLOBALLY (all_land) per model×pair, then applied
# to each domain subset, so all zones share the same outlier thresholds.
# Core-zone samples are omitted: they are not used by the branch detectors.
JOBS = []
for run in RUNS:
    t = run['clim']
    run_meta = {
        'model': run['model'],
        'experiment': run['experiment'],
        'member': run['member'],
        'grid': run['grid'],
    }

    global_bounds = {}
    for x_var, y_var, pair_label in VARIABLE_PAIRS:
        if x_var not in t.columns or y_var not in t.columns:
            continue
        x_all = np.asarray(t[x_var], dtype=float)
        y_all = np.asarray(t[y_var], dtype=float)
        finite = np.isfinite(x_all) & np.isfinite(y_all)
        xf, yf = x_all[finite], y_all[finite]
        if len(xf) == 0:
            global_bounds[(x_var, y_var)] = (np.nan, np.nan, np.nan, np.nan)
        else:
            x_lo, x_hi = np.quantile(xf, [TRIM_LOWER_QUANTILE, TRIM_UPPER_QUANTILE])
            y_lo, y_hi = np.quantile(yf, [TRIM_LOWER_QUANTILE, TRIM_UPPER_QUANTILE])
            global_bounds[(x_var, y_var)] = (
                float(x_lo), float(x_hi), float(y_lo), float(y_hi),
            )

    for domain_key, domain_label, domain_filter in DOMAINS:
        subset = t.loc[domain_filter(t)] if domain_filter is not None else t
        for x_var, y_var, pair_label in VARIABLE_PAIRS:
            if (x_var, y_var) not in global_bounds:
                continue
            x_arr = np.asarray(subset[x_var], dtype=float)
            y_arr = np.asarray(subset[y_var], dtype=float)
            finite = np.isfinite(x_arr) & np.isfinite(y_arr)
            x_raw = x_arr[finite]
            y_raw = y_arr[finite]
            raw_n = len(x_raw)
            bounds = global_bounds[(x_var, y_var)]
            x_lo, x_hi, y_lo, y_hi = bounds
            keep = (
                (x_raw >= x_lo) & (x_raw <= x_hi)
                & (y_raw >= y_lo) & (y_raw <= y_hi)
            )
            for version in ['raw', 'trimmed']:
                if version == 'trimmed':
                    xv, yv = x_raw[keep], y_raw[keep]
                else:
                    xv, yv = x_raw, y_raw
                JOBS.append((
                    xv.copy(), yv.copy(),
                    x_var, y_var, domain_key, 'all',
                    version, raw_n, bounds, run_meta,
                ))


def _worker(job):
    x, y, x_name, y_name, domain, sample, version, raw_n, bounds, meta = job
    res = run_association_gate(
        x, y, x_name, y_name, domain, sample, version, raw_n, bounds,
    )
    res.update(meta)
    return res


N_WORKERS = max(1, (os.cpu_count() or 4) - 1)
print(f'Running {len(JOBS)} MIC/dCor gates on {N_WORKERS} threads ...')
_t0 = _time.perf_counter()

ALL_RESULTS = []
_done = 0
with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
    futures = [pool.submit(_worker, job) for job in JOBS]
    for fut in as_completed(futures):
        ALL_RESULTS.append(fut.result())
        _done += 1
        if _done % 50 == 0 or _done == len(JOBS):
            print(f'  {_done}/{len(JOBS)} done')

_elapsed = _time.perf_counter() - _t0
print(f'Finished in {_elapsed:.1f}s')

ALL_RESULTS.sort(key=lambda r: (
    r['model'], r['domain'], r['pair'], r['sample'], r['outlier_version']
))

RESULTS_DF = pd.DataFrame(ALL_RESULTS)
print(f'\nTotal gate panels: {len(RESULTS_DF)}')
for version in ['raw', 'trimmed']:
    n = int((RESULTS_DF['outlier_version'] == version).sum())
    print(f'  {version}: {n} cases')

print('\nGate pass counts (intermediate or strong):')
gate_view = RESULTS_DF.copy()
gate_view['both_pass'] = gate_view['mic_pass'] & gate_view['dcor_pass']
gate_view['either_pass'] = gate_view['mic_pass'] | gate_view['dcor_pass']
print(
    gate_view.groupby(['pair', 'outlier_version'])[
        ['mic_pass', 'dcor_pass', 'both_pass', 'either_pass']
    ].sum().astype(int).to_string()
)


## 6. Pair analysis — RAW / 按变量对（raw）

> Disabled. Unused for the current MIC/dCor gate → branch workflow.

**中文说明：** 已注释。当前主流程不跑这一节。


In [4]:
%%script --no-raise-error false
# Disabled: unused for the MIC/dCor gate → branch workflow (sections 14–15).
from statsmodels.nonparametric.smoothers_lowess import lowess as _lowess


def style_axes(ax):
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('#30343B')
        spine.set_linewidth(1.0)
    ax.tick_params(
        direction='out', length=4.0, width=0.9,
        color='#30343B', labelsize=11,
    )
    ax.grid(True, color='#D3D3D3', linewidth=0.6, alpha=0.7, zorder=0)


UNIT_LABELS = {
    'P': 'Precipitation, P (mm yr⁻¹)',
    'ET': 'Evapotranspiration, ET (mm yr⁻¹)',
    'Q': 'Total runoff, Q (mm yr⁻¹)',
    'hfls': 'Latent heat flux (W m⁻²)',
    'hfss': 'Sensible heat flux (W m⁻²)',
    'tran': 'Transpiration (mm yr⁻¹)',
    'evspsblsoi': 'Soil evaporation (mm yr⁻¹)',
    'mrros': 'Surface runoff (mm yr⁻¹)',
    'mrso': 'Total soil moisture (kg m⁻²)',
    'mrsos': 'Topsoil moisture (kg m⁻²)',
    'lai': 'Leaf area index (m² m⁻²)',
    'tas': 'Near-surface temperature (K)',
    'prsn': 'Snowfall (mm yr⁻¹)',
    'rlds': 'Downward LW radiation (W m⁻²)',
    'rlus': 'Upward LW radiation (W m⁻²)',
    'rsds': 'Downward SW radiation (W m⁻²)',
    'rsus': 'Upward SW radiation (W m⁻²)',
}
PAIR_SLUGS = {label: f'{x}_{y}' for x, y, label in VARIABLE_PAIRS}


def draw_lowess_fit(ax, x_values, y_values, frac=0.3, it=3):
    x_arr = np.asarray(x_values, dtype=float)
    y_arr = np.asarray(y_values, dtype=float)
    finite = np.isfinite(x_arr) & np.isfinite(y_arr)
    xf, yf = x_arr[finite], y_arr[finite]
    if len(xf) < 10:
        return False
    result = _lowess(yf, xf, frac=frac, it=it, return_sorted=True)
    ax.plot(
        result[:, 0], result[:, 1], color='#B12A68', linewidth=1.2,
        linestyle='-', alpha=0.4, zorder=5,
    )
    return True


def draw_density_kde(ax, x_values, y_values, cmap='Reds', n_levels=12, n_grid=100):
    """KDE contour density plot with scatter overlay."""
    from scipy.stats import gaussian_kde
    x_arr = np.asarray(x_values, dtype=float)
    y_arr = np.asarray(y_values, dtype=float)
    finite = np.isfinite(x_arr) & np.isfinite(y_arr)
    xf, yf = x_arr[finite], y_arr[finite]
    if len(xf) < 20:
        ax.scatter(xf, yf, s=5, c='k', alpha=0.4, rasterized=True)
        return
    xy = np.vstack([xf, yf])
    try:
        kde = gaussian_kde(xy)
    except np.linalg.LinAlgError:
        ax.scatter(xf, yf, s=5, c='k', alpha=0.4, rasterized=True)
        return
    xi = np.linspace(xf.min(), xf.max(), n_grid)
    yi = np.linspace(yf.min(), yf.max(), n_grid)
    Xi, Yi = np.meshgrid(xi, yi)
    Zi = kde(np.vstack([Xi.ravel(), Yi.ravel()])).reshape(Xi.shape)
    Zi_log = np.log10(np.maximum(Zi, 1e-20))
    ax.contourf(Xi, Yi, Zi_log, levels=n_levels, cmap=cmap, alpha=0.85, zorder=1)
    ax.scatter(xf, yf, s=2, c='k', alpha=0.15, edgecolors='none', rasterized=True, zorder=2)


def draw_density_hexbin(ax, x_values, y_values, cmap='Reds', gridsize=20):
    """Hexbin density plot."""
    from matplotlib.colors import LogNorm
    x_arr = np.asarray(x_values, dtype=float)
    y_arr = np.asarray(y_values, dtype=float)
    finite = np.isfinite(x_arr) & np.isfinite(y_arr)
    xf, yf = x_arr[finite], y_arr[finite]
    if len(xf) < 20:
        ax.scatter(xf, yf, s=5, c='k', alpha=0.4, rasterized=True)
        return
    ax.hexbin(xf, yf, gridsize=gridsize, cmap=cmap, mincnt=1, norm=LogNorm(), zorder=1)


# Full names for figure titles (abbreviation → readable name)
PAIR_FULL_NAMES = {}
for _x, _y, _label in VARIABLE_PAIRS:
    _x_name = UNIT_LABELS.get(_x, _x).split('(')[0].strip().rstrip(',')
    _y_name = UNIT_LABELS.get(_y, _y).split('(')[0].strip().rstrip(',')
    PAIR_FULL_NAMES[_label] = f'{_x_name} → {_y_name}'
SUMMARY_COLUMNS = [
    'model', 'domain', 'n_cells', 'n_removed', 'removed_fraction',
    'mic', 'pearson_r_descriptive', 'group', 'path_trace',
]
S4_OUTPUT_DIR = CASE_DIR / 'output' / 'S4'
S4_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR = S4_OUTPUT_DIR / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


def assign_model_colors(model_names):
    ordered = list(dict.fromkeys(model_names))
    return {
        name: MODEL_COLOR_CYCLE[i % len(MODEL_COLOR_CYCLE)]
        for i, name in enumerate(ordered)
    }


MODEL_COLORS = assign_model_colors([run['model'] for run in RUNS])
print('Model colours:')
for name, color in MODEL_COLORS.items():
    print(f'  {name}: {color}')


def padded_limits(values, pad_fraction=0.04):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return (-1.0, 1.0)
    lower, upper = float(values.min()), float(values.max())
    span = upper - lower
    if not np.isfinite(span) or span == 0:
        span = max(abs(lower), 1.0)
    pad = pad_fraction * span
    return lower - pad, upper + pad


def lookup_result_row(model, domain_key, x_var, y_var, version):
    rows = RESULTS_DF.loc[
        (RESULTS_DF['model'] == model)
        & (RESULTS_DF['domain'] == domain_key)
        & (RESULTS_DF['sample'] == 'all')
        & (RESULTS_DF['x_var'] == x_var)
        & (RESULTS_DF['y_var'] == y_var)
        & (RESULTS_DF['outlier_version'] == version)
    ]
    if len(rows) != 1:
        return None
    return rows.iloc[0]


def collect_overlay_series(x_var, y_var, domain_key, domain_filter, version):
    """Points for every model that has finite cells in this domain."""
    series = []
    for run in RUNS:
        table = run['clim']
        subset = table if domain_filter is None else table.loc[domain_filter(table)]
        if subset.empty:
            continue
        prepared = prepare_pair_versions(subset[x_var], subset[y_var])
        x_raw, y_raw = prepared['raw']
        if len(x_raw) == 0:
            continue
        x_vals, y_vals = prepared[version]
        keep = prepared['trim_keep']
        series.append({
            'model': run['model'],
            'x': x_vals,
            'y': y_vals,
            'x_raw': x_raw,
            'y_raw': y_raw,
            'trim_keep': keep,
            'xlim_src': x_raw,
            'ylim_src': y_raw,
            'result': lookup_result_row(
                run['model'], domain_key, x_var, y_var, version
            ),
        })
    return series


def pair_summary_table(pair_label, version):
    table = RESULTS_DF.loc[
        (RESULTS_DF['pair'] == pair_label)
        & (RESULTS_DF['outlier_version'] == version)
        & (RESULTS_DF['sample'] == 'all'),
        [c for c in SUMMARY_COLUMNS if c in RESULTS_DF.columns],
    ].copy()
    table = table.sort_values(['domain', 'model'])
    display_table = table.astype(object).where(pd.notna(table), '-').replace('N/A', '-')
    print(
        f'\n=== {version.upper()} · {pair_label} · all models '
        f'({len(display_table)} rows) ==='
    )
    display(
        display_table.style.set_properties(
            **{'text-align': 'left', 'white-space': 'nowrap', 'font-size': '12px'}
        )
    )
    out = S4_OUTPUT_DIR / f'S4_summary_{PAIR_SLUGS[pair_label]}_{version}.csv'
    if WRITE_OUTPUTS:
        table.to_csv(out, index=False)
        print(f'saved {out}')
    return table


def draw_pair_overlay_figure(x_var, y_var, pair_label, version):
    n_domains = len(DOMAINS)
    n_cols = 4
    n_rows = int(np.ceil(n_domains / n_cols))
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(5 * n_cols, 4 * n_rows),
        constrained_layout=True,
    )
    axes = np.atleast_1d(axes).ravel()

    legend_handles = []
    seen_models = []

    for idx, (domain_key, domain_label, domain_filter) in enumerate(DOMAINS):
        ax = axes[idx]
        series = collect_overlay_series(
            x_var, y_var, domain_key, domain_filter, version
        )
        x_pool, y_pool = [], []
        for item in series:
            x_pool.append(item['xlim_src'])
            y_pool.append(item['ylim_src'])
            color = MODEL_COLORS[item['model']]
            if version == 'trimmed':
                dropped = ~np.asarray(item['trim_keep'], dtype=bool)
                if dropped.any():
                    ax.scatter(
                        item['x_raw'][dropped], item['y_raw'][dropped],
                        marker='x', s=18, c=color, alpha=0.80,
                        linewidths=0.7, zorder=1, rasterized=True,
                    )
            if len(item['x']) > 0:
                ax.scatter(
                    item['x'], item['y'],
                    s=10, c=color, alpha=0.3,
                    edgecolors='none', rasterized=True, zorder=2,
                    label=item['model'],
                )
                draw_lowess_fit(ax, item['x'], item['y'])
            if item['model'] not in seen_models:
                seen_models.append(item['model'])
                legend_handles.append(
                    Line2D(
                        [0], [0], marker='o', color='none',
                        markerfacecolor=color, markeredgecolor='none',
                        markersize=10, alpha=0.6,
                        label=item['model'],
                    )
                )
        if series:
            ax.set_xlim(padded_limits(np.concatenate(x_pool)))
            ax.set_ylim(padded_limits(np.concatenate(y_pool)))
        else:
            ax.text(
                0.5, 0.5, 'No models with data',
                transform=ax.transAxes, ha='center', va='center',
                fontsize=13, color='#6B7280',
            )
        ax.set_title(f'{domain_label}  ({domain_key})', fontsize=14, pad=8)
        ax.set_xlabel(UNIT_LABELS[x_var], fontsize=13)
        ax.set_ylabel(UNIT_LABELS[y_var], fontsize=13)
        style_axes(ax)
        panel_letter = chr(ord('a') + idx)
        ax.text(
            0.02, 0.98, panel_letter,
            transform=ax.transAxes, va='top', ha='left',
            fontsize=13, fontweight='bold', color='#222222',
        )

    for ax in axes[n_domains:]:
        ax.set_visible(False)

    fig.suptitle(
        f'{version.upper()}  ·  {PAIR_FULL_NAMES.get(pair_label, pair_label)}  ·  all models with data',
        fontsize=18, fontweight='semibold', y=1.05,
    )
    if legend_handles:
        fig.legend(
            handles=legend_handles,
            loc='lower center',
            ncol=min(5, len(legend_handles)),
            bbox_to_anchor=(0.5, -0.14),
            frameon=True,
            fancybox=True,
            framealpha=0.5,
            edgecolor='none',
            fontsize=12,
            markerscale=1.3,
            title='Model (colour)',
            title_fontsize=13,
        )
        if version == 'trimmed':
            fig.legend(
                handles=[
                    Line2D(
                        [0], [0], marker='x', color='#444444',
                        linestyle='none', markersize=8,
                        label='Removed (outside 1st–99th percentile on x or y)',
                    )
                ],
                loc='lower center',
                bbox_to_anchor=(0.5, -0.20),
                frameon=True,
                fancybox=True,
                framealpha=0.5,
                edgecolor='none',
                fontsize=11,
            )
    if WRITE_OUTPUTS:
        path = FIGURE_DIR / f'S4_overlay_{PAIR_SLUGS[pair_label]}_{version}.png'
        fig.savefig(path, dpi=180, bbox_inches='tight')
        print(f'saved {path}')
    plt.show()
    plt.close(fig)


# print('RAW pair analysis')
# print('=' * 72)
# for x_var, y_var, pair_label in VARIABLE_PAIRS:
#     print('\n' + '#' * 72)
#     print(f'RAW · {pair_label}')
#     print('#' * 72)
#     pair_summary_table(pair_label, 'raw')
#     draw_pair_overlay_figure(x_var, y_var, pair_label, 'raw')



SHAPE_FACECOLORS = {
    'Linear':       '#E8F5E9',
    'Saturating':   '#FFF3E0',
    'Accelerating': '#E3F2FD',
    'S-shape':      '#F3E5F5',
    'Flat/Weak':    '#F5F5F5',
    'U-shape':      '#FFF9C4',
    'Arch-shape':   '#FFE0B2',
    'Complex':      '#FFEBEE',
    'Insufficient data': '#FFFFFF',
}


def extract_lowess_shape(x_data, y_data, frac=0.3, it=3):
    """Fit LOWESS and extract shape features from the smoothed curve."""
    from scipy.signal import find_peaks

    x_arr = np.asarray(x_data, dtype=float)
    y_arr = np.asarray(y_data, dtype=float)
    finite = np.isfinite(x_arr) & np.isfinite(y_arr)
    xf, yf = x_arr[finite], y_arr[finite]
    if len(xf) < 20:
        return None

    result = _lowess(yf, xf, frac=frac, it=it, return_sorted=True)
    xs, ys = result[:, 0], result[:, 1]

    y_range = ys.max() - ys.min()
    x_range = xs.max() - xs.min()
    if x_range == 0 or y_range == 0:
        return None

    # Power-law fit on LOWESS curve: y = a * x^b
    # Fit in log-log space, evaluate R² in original space
    power_b = np.nan
    power_r2 = np.nan
    abs_b_minus_1 = np.nan
    pos_mask = (xs > 0) & (ys > 0)
    if pos_mask.sum() >= 10:
        log_x = np.log(xs[pos_mask])
        log_y = np.log(ys[pos_mask])
        coef = np.polyfit(log_x, log_y, 1)
        power_b = coef[0]
        abs_b_minus_1 = abs(power_b - 1.0)
        # R² in original space: power law prediction vs LOWESS curve
        y_pred = np.exp(np.polyval(coef, log_x))
        y_actual = ys[pos_mask]
        ss_res = np.sum((y_actual - y_pred) ** 2)
        ss_tot = np.sum((y_actual - y_actual.mean()) ** 2)
        power_r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else 1.0

    # First derivative
    dx = np.diff(xs)
    dy = np.diff(ys)
    mask = dx > 0
    dx, dy = dx[mask], dy[mask]
    if len(dx) < 5:
        return None
    slope = dy / dx

    # Second derivative
    x_mid = 0.5 * (xs[:-1][mask] + xs[1:][mask])
    dx2 = np.diff(x_mid)
    dslope = np.diff(slope)
    mask2 = dx2 > 0
    curvature = dslope[mask2] / dx2[mask2] if mask2.any() else np.array([])

    norm_slope = slope * (x_range / y_range)

    frac_positive = (slope > 0).mean()
    frac_negative = (slope < 0).mean()
    mean_norm_slope = norm_slope.mean()
    slope_std = norm_slope.std()

    n_edge = max(3, int(0.2 * len(slope)))
    slope_start = float(np.median(slope[:n_edge]))
    slope_end = float(np.median(slope[-n_edge:]))

    if len(curvature) > 0:
        frac_concave = (curvature > 0).mean()
        mean_curvature = float(np.mean(curvature * (x_range**2 / y_range)))
        half = len(curvature) // 2
        curv_first_half = curvature[:half].mean() if half > 0 else 0
        curv_second_half = curvature[half:].mean()
    else:
        frac_concave = np.nan
        mean_curvature = np.nan
        curv_first_half = 0
        curv_second_half = 0

    net_change = (ys[-1] - ys[0]) / y_range if y_range > 0 else 0

    # Turning points via find_peaks with prominence
    amplitude = np.ptp(ys)
    n_peaks, n_troughs = 0, 0
    if amplitude > 1e-10:
        prom = max(0.10 * amplitude, 1e-10)
        min_dist = max(5, int(0.08 * len(ys)))
        peaks, _ = find_peaks(ys, prominence=prom, distance=min_dist)
        troughs, _ = find_peaks(-ys, prominence=prom, distance=min_dist)
        n_peaks = len(peaks)
        n_troughs = len(troughs)

    return {
        'frac_positive_slope': frac_positive,
        'frac_negative_slope': frac_negative,
        'mean_norm_slope': mean_norm_slope,
        'slope_std': slope_std,
        'slope_start': slope_start,
        'slope_end': slope_end,
        'frac_concave': frac_concave,
        'mean_curvature': mean_curvature,
        'net_change': net_change,
        'curv_first_half': curv_first_half,
        'curv_second_half': curv_second_half,
        'power_b': power_b,
        'power_r2': power_r2,
        'abs_b_minus_1': abs_b_minus_1,
        'n_peaks': n_peaks,
        'n_troughs': n_troughs,
    }


def classify_lowess_shape(feat, flat_thresh=0.15):
    """Assign a shape label based on extracted LOWESS features."""
    if feat is None:
        return 'Insufficient data'

    net = feat['net_change']
    frac_pos = feat['frac_positive_slope']
    frac_neg = feat['frac_negative_slope']
    frac_concave = feat['frac_concave']
    power_b = feat['power_b']
    power_r2 = feat['power_r2']
    abs_b_1 = feat['abs_b_minus_1']
    n_peaks = feat['n_peaks']
    n_troughs = feat['n_troughs']
    n_tp = n_peaks + n_troughs

    # 1. Flat / Weak
    if abs(net) < flat_thresh:
        if n_peaks == 1 and n_troughs == 0:
            return 'Arch-shape'
        if n_troughs == 1 and n_peaks == 0:
            return 'U-shape'
        return 'Flat/Weak'

    # 2. Power-law fit first — if it fits well, use b to classify
    has_power = np.isfinite(power_b) and np.isfinite(power_r2)
    is_monotone_inc = frac_pos >= 0.80
    is_monotone_dec = frac_neg >= 0.80
    is_monotone = is_monotone_inc or is_monotone_dec

    if has_power and power_r2 >= 0.50 and is_monotone:
        if abs_b_1 < 0.15:
            return 'Linear'
        if power_b > 1.15:
            return 'Accelerating'
        if power_b < 0.85:
            # Cross-check: if curvature says concave-up, trust curvature
            if is_monotone_inc and frac_concave > 0.55:
                return 'Accelerating'
            return 'Saturating'

    # 3. Power-law fit poor — check turning points
    if n_tp >= 3:
        return 'Complex'
    if n_peaks == 1 and n_troughs == 0:
        return 'Arch-shape'
    if n_troughs == 1 and n_peaks == 0:
        return 'U-shape'
    if n_tp == 2:
        return 'S-shape'

    # 4. Monotone but power-law R² low — use curvature only if R² is moderate
    if is_monotone:
        if not has_power or power_r2 < 0.50:
            return 'Complex'
        if is_monotone_inc and frac_concave < 0.40:
            return 'Saturating'
        if is_monotone_dec and frac_concave > 0.60:
            return 'Saturating'
        if is_monotone_inc and frac_concave > 0.60:
            return 'Accelerating'
        if is_monotone_dec and frac_concave < 0.40:
            return 'Accelerating'
        return 'Linear'

    # 5. Non-monotone without clear turning points
    c1 = feat['curv_first_half']
    c2 = feat['curv_second_half']
    if c1 * c2 < 0:
        return 'S-shape'

    return 'Complex' 


## 6b. Pair analysis — TRIMMED / 按变量对（trimmed）

Same tables and multi-model overlays as the raw section, for the two selected pairs.

**中文说明：** 与 raw 部分相同的表和图，仅针对两个变量对的 trimmed 版本。

In [5]:
# print('TRIMMED pair analysis')
# print('=' * 72)
# for x_var, y_var, pair_label in VARIABLE_PAIRS:
#     print('\n' + '#' * 72)
#     print(f'TRIMMED · {pair_label}')
#     print('#' * 72)
#     pair_summary_table(pair_label, 'trimmed')
#     draw_pair_overlay_figure(x_var, y_var, pair_label, 'trimmed')


## 6c. Per-model scatter plots — RAW / 单模型散点（raw）

Two variable-pair rows (P→Q, mrros→Q) × domain columns per model.

**中文说明：** 每个模型两行（P→Q, mrros→Q）× 分域列的散点图。

In [6]:
%%script --no-raise-error false
# Disabled: unused for the MIC/dCor gate → branch workflow (sections 14–15).
N_EQUAL_COUNT_BINS = 10  # 等频分箱数量（用于散点图可视化）


def equal_count_bin_medians(x, y, n_bins=N_EQUAL_COUNT_BINS):
    frame = pd.DataFrame({'x': x, 'y': y}).replace(
        [np.inf, -np.inf], np.nan
    ).dropna()
    if len(frame) < 2 or frame['x'].nunique() < 2:
        return pd.DataFrame(columns=['x', 'y', 'n'])
    q = min(int(n_bins), int(frame['x'].nunique()))
    frame['bin'] = pd.qcut(frame['x'], q=q, duplicates='drop')
    curve = frame.groupby('bin', observed=True).agg(
        x=('x', 'median'), y=('y', 'median'), n=('y', 'size')
    ).reset_index(drop=True)
    return curve.sort_values('x')




def metric_text(row, column):
    value = row.get(column, np.nan)
    return f'{value:.3f}' if pd.notna(value) else '—'


def display_class(group):
    return {
        'No Global Relationship': 'No global relationship',
    }.get(str(group), str(group))


def classification_facecolor(group):
    if group is None:
        return '#FFFFFF'
    try:
        if isinstance(group, float) and np.isnan(group):
            return '#FFFFFF'
    except TypeError:
        pass
    key = str(group)
    if key in CLASS_FACECOLORS:
        return CLASS_FACECOLORS[key]
    return CLASS_FACECOLORS.get(display_class(key), '#FFFFFF')


def format_panel_annotation(result_row, version, raw_row=None):
    """Return (class_name, detail_text). Class is drawn bold; no 'Final class' prefix."""
    if result_row is None:
        return 'No result', ''
    final_class = display_class(result_row['group'])
    if final_class in {'', 'nan', 'None'}:
        final_class = 'No result'
    details = (
        f"MIC={metric_text(result_row, 'mic')}  "
        f"dcor={metric_text(result_row, 'dcor')}  "
        f"r={metric_text(result_row, 'pearson_r_descriptive')}\n"
        f"n={int(result_row['n_cells'])}"
    )
    if pd.notna(result_row.get('power_r2', np.nan)):
        details += (
            f"  R²={float(result_row['power_r2']):.3f}"
            f"  |b−1|={float(result_row.get('abs_b_minus_1', np.nan)):.3f}"
        )
    if version == 'trimmed':
        details += (
            f"\nrem={int(result_row['n_removed'])} "
            f"({100 * float(result_row['removed_fraction']):.1f}%)"
        )
        if raw_row is not None and raw_row['group'] != result_row['group']:
            details += (
                f"  {display_class(raw_row['group'])} → {final_class}"
            )
    return final_class, details


def draw_class_annotation(ax, class_name, details, *, fontsize=9.2, y=0.96):
    """Bold class name, then metrics — one box, no overlapping text."""
    from matplotlib.offsetbox import AnnotationBbox, TextArea, VPacker

    title = TextArea(
        class_name,
        textprops={
            'fontsize': fontsize + 0.8,
            'fontweight': 'bold',
            'color': '#111111',
            'family': 'DejaVu Sans',
        },
    )
    children = [title]
    if details:
        children.append(
            TextArea(
                details,
                textprops={
                    'fontsize': fontsize,
                    'fontweight': 'normal',
                    'color': '#202020',
                    'family': 'DejaVu Sans',
                },
            )
        )
    pack = VPacker(children=children, align='left', pad=0, sep=3)
    box = AnnotationBbox(
        pack,
        (0.03, y),
        xycoords='axes fraction',
        box_alignment=(0.0, 1.0),
        frameon=True,
        pad=0.30,
        bboxprops={
            'boxstyle': 'round,pad=0.35',
            'facecolor': 'white',
            'edgecolor': '#C9C9C9',
            'alpha': 0.92,
        },
        zorder=8,
    )
    ax.add_artist(box)


def scatter_points(ax, x_data, y_data, domain_key, groups,
                   dropped=None, dropped_groups=None):
    if dropped is not None:
        xd, yd = dropped
        if len(xd) > 0:
            if domain_key == 'non_ice_land' and dropped_groups is not None:
                dcolors = [ZONE_COLORS.get(str(z), '#B0B0B0') for z in dropped_groups]
                ax.scatter(
                    xd, yd, marker='x', s=16, c=dcolors, alpha=0.85,
                    linewidths=0.7, zorder=1, rasterized=True,
                )
            else:
                ax.scatter(
                    xd, yd, marker='x', s=16,
                    color=ZONE_COLORS.get(domain_key, '#555555'),
                    alpha=0.85, linewidths=0.7, zorder=1, rasterized=True,
                )
    if len(x_data) == 0:
        if dropped is None or len(dropped[0]) == 0:
            ax.text(
                0.5, 0.5, 'Unavailable', transform=ax.transAxes,
                ha='center', va='center', fontsize=12, color='#6B7280',
            )
        return
    if domain_key == 'non_ice_land' and groups is not None:
        colors = [ZONE_COLORS.get(str(z), '#B0B0B0') for z in groups]
        ax.scatter(
            x_data, y_data, s=7, c=colors, alpha=0.55,
            edgecolors='none', rasterized=True, zorder=2,
        )
        return
    color = ZONE_COLORS.get(domain_key, '#555555')
    size = 8 if domain_key == 'LI' else 7
    ax.scatter(
        x_data, y_data, s=size, color=color, alpha=0.55,
        edgecolors='none', rasterized=True, zorder=2,
    )


def build_panel_payloads(run):
    t = run['clim']
    payloads = {}
    for x_var, y_var, pair_label in VARIABLE_PAIRS:
        for domain_key, domain_label, domain_filter in DOMAINS:
            subset = t.loc[domain_filter(t)] if domain_filter is not None else t
            prepared = prepare_pair_versions(
                subset[x_var], subset[y_var],
                groups=subset['analysis_zone'] if 'analysis_zone' in subset else None,
            )
            x_raw, y_raw = prepared['raw']
            result_rows = RESULTS_DF.loc[
                (RESULTS_DF['model'] == run['model'])
                & (RESULTS_DF['experiment'] == run['experiment'])
                & (RESULTS_DF['member'] == run['member'])
                & (RESULTS_DF['grid'] == run['grid'])
                & (RESULTS_DF['domain'] == domain_key)
                & (RESULTS_DF['sample'] == 'all')
                & (RESULTS_DF['x_var'] == x_var)
                & (RESULTS_DF['y_var'] == y_var)
            ]
            payloads[(pair_label, domain_key)] = {
                'x_var': x_var,
                'y_var': y_var,
                'pair_label': pair_label,
                'domain_key': domain_key,
                'domain_label': domain_label,
                'prepared': prepared,
                'result_rows': result_rows,
                'xlim': padded_limits(x_raw),
                'ylim': padded_limits(y_raw),
            }
    return payloads


def draw_relationship_figure(run, version, panel_payloads):
    n_rows = len(VARIABLE_PAIRS)
    n_cols = len(DOMAINS)
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(3.9 * n_cols, 4.15 * n_rows),
        constrained_layout=True, squeeze=False,
    )

    for row_i, (x_var, y_var, pair_label) in enumerate(VARIABLE_PAIRS):
        for col_i, (domain_key, domain_label, _) in enumerate(DOMAINS):
            ax = axes[row_i, col_i]
            payload = panel_payloads[(pair_label, domain_key)]
            x_data, y_data = payload['prepared'][version]
            groups = payload['prepared'][f'{version}_groups']
            dropped = None
            dropped_groups = None
            if version == 'trimmed':
                keep = np.asarray(payload['prepared']['trim_keep'], dtype=bool)
                x_raw, y_raw = payload['prepared']['raw']
                dropped = (x_raw[~keep], y_raw[~keep])
                g_raw = payload['prepared']['raw_groups']
                if g_raw is not None:
                    dropped_groups = g_raw[~keep]
            scatter_points(
                ax, x_data, y_data, domain_key, groups,
                dropped=dropped, dropped_groups=dropped_groups,
            )

            bin_curve = equal_count_bin_medians(x_data, y_data)
            if len(bin_curve) >= 2:
                ax.plot(
                    bin_curve['x'], bin_curve['y'],
                    color='white', linewidth=4.2, zorder=3,
                )
                ax.plot(
                    bin_curve['x'], bin_curve['y'],
                    color='#20252B', linewidth=2.0, marker='o',
                    markersize=3.4, markerfacecolor='white',
                    markeredgecolor='#20252B', markeredgewidth=0.8,
                    zorder=4,
                )
            draw_lowess_fit(ax, x_data, y_data)

            version_rows = payload['result_rows'].loc[
                payload['result_rows']['outlier_version'] == version
            ]
            raw_rows = payload['result_rows'].loc[
                payload['result_rows']['outlier_version'] == 'raw'
            ]
            if len(version_rows) == 1:
                result_row = version_rows.iloc[0]
                raw_row = raw_rows.iloc[0] if len(raw_rows) == 1 else None
                class_name, details = format_panel_annotation(
                    result_row, version, raw_row=raw_row
                )
            else:
                class_name, details = 'No result', ''
            draw_class_annotation(ax, class_name, details, fontsize=9.2)
            ax.set_xlim(payload['xlim'])
            ax.set_ylim(payload['ylim'])
            if row_i == 0:
                ax.set_title(
                    f'{domain_label}\n({domain_key})',
                    fontsize=12, fontweight='semibold', pad=8,
                )
            ax.set_xlabel(UNIT_LABELS[x_var], fontsize=10)
            ax.set_ylabel(
                UNIT_LABELS[y_var] if col_i == 0 else '',
                fontsize=10,
            )
            if col_i == 0:
                ax.text(
                    -0.24, 0.5, pair_label,
                    transform=ax.transAxes, rotation=90,
                    va='center', ha='center',
                    fontsize=13, fontweight='bold', color='#222222',
                )
            ax.text(
                0.98, 0.02, chr(ord('a') + row_i * n_cols + col_i),
                transform=ax.transAxes, va='bottom', ha='right',
                fontsize=11, fontweight='bold', color='#444444',
            )
            style_axes(ax)
            face = '#FFFFFF'
            if len(version_rows) == 1:
                face = classification_facecolor(version_rows.iloc[0]['group'])
            ax.set_facecolor(face)

    fig.suptitle(
        f"{version.upper()} per-model  ·  {run['model']}  ·  "
        f"{START_YEAR}–{END_YEAR}",
        fontsize=16, fontweight='semibold',
    )
    legend_handles = [
        Line2D(
            [0], [0], color='#20252B', linewidth=2.0, marker='o',
            markersize=4, markerfacecolor='white', alpha=0.6,
            label=f'Median trend ({N_EQUAL_COUNT_BINS} equal-count bins)',
        ),
        Line2D(
            [0], [0], color='#B12A68', linewidth=1.8,
            linestyle='-', alpha=0.6,
            label='LOWESS fit',
        ),
        Line2D(
            [0], [0], marker='x', color='#444444', linestyle='none',
            markersize=8, alpha=0.6,
            label='Removed in trimmed (outside 1st–99th percentile)',
        ),
    ]
    for key, label in ZONE_LEGEND:
        legend_handles.append(
            Line2D(
                [0], [0], marker='o', color='none',
                markerfacecolor=ZONE_COLORS[key], markeredgecolor='none',
                markersize=8, alpha=0.6, label=label,
            )
        )
    fig.legend(
        handles=legend_handles, loc='lower center', ncol=7,
        bbox_to_anchor=(0.5, -0.06), frameon=True, fancybox=True, framealpha=0.5, edgecolor='none', fontsize=11,
    )
    if WRITE_OUTPUTS:
        fig.savefig(
            FIGURE_DIR / f"S4_relationships_{run['model']}_{version}.png",
            dpi=180, bbox_inches='tight',
        )
    plt.show()
    plt.close(fig)


# print('Per-model RAW scatter plots')
# print('=' * 72)
# for run in RUNS:
#     payloads = build_panel_payloads(run)
#     run['panel_payloads'] = payloads
#     draw_relationship_figure(run, 'raw', payloads)


## 6d. Per-model scatter plots — TRIMMED / 单模型散点（trimmed）

Same layout as 6c, trimmed version.

**中文说明：** 与6c相同布局的trimmed版本。

In [7]:
# print('Per-model TRIMMED scatter plots')
# print('=' * 72)
# for run in RUNS:
#     payloads = run.get('panel_payloads')
#     if payloads is None:
#         payloads = build_panel_payloads(run)
#         run['panel_payloads'] = payloads
#     draw_relationship_figure(run, 'trimmed', payloads)


## 6e. Model × zone grid — RAW / 模型×分区总图（raw）

> Disabled. Unused for the current MIC/dCor gate → branch workflow.

**中文说明：** 已注释。当前主流程不跑这一节。


In [8]:
%%script --no-raise-error false
# Disabled: unused for the MIC/dCor gate → branch workflow (sections 14–15).
def _ensure_payloads(run):
    payloads = run.get('panel_payloads')
    if payloads is None:
        payloads = build_panel_payloads(run)
        run['panel_payloads'] = payloads
    return payloads


def _raw_count(run, pair_label, domain_key):
    payload = _ensure_payloads(run).get((pair_label, domain_key))
    if payload is None:
        return 0
    return len(payload['prepared']['raw'][0])


def column_limits_for_pair(pair_label, domain_key, runs, version='raw'):
    xs, ys = [], []
    for run in runs:
        payload = _ensure_payloads(run).get((pair_label, domain_key))
        if payload is None:
            continue
        xv, yv = payload['prepared'][version]
        if len(xv) == 0:
            continue
        xs.append(xv)
        ys.append(yv)
    if not xs:
        return (-1.0, 1.0), (-1.0, 1.0)
    return padded_limits(np.concatenate(xs)), padded_limits(np.concatenate(ys))


def select_grid_runs(pair_label):
    keep, skipped = [], []
    for run in RUNS:
        n_any = sum(
            _raw_count(run, pair_label, domain_key)
            for domain_key, _, _ in DOMAINS
        )
        n_not_all_land = sum(
            _raw_count(run, pair_label, domain_key)
            for domain_key, _, _ in DOMAINS
            if domain_key != 'all_land'
        )
        if n_not_all_land == 0:
            skipped.append(run['model'])
            continue
        if n_any == 0:
            skipped.append(run['model'])
            continue
        keep.append(run)
    if skipped:
        print(f'  omit empty models from overview: {", ".join(skipped)}')
    return keep


def select_grid_domains(pair_label, runs):
    keep = []
    for domain in DOMAINS:
        if any(_raw_count(run, pair_label, domain[0]) > 0 for run in runs):
            keep.append(domain)
    return keep


SHAPE_FACECOLORS = {
    'Linear':       '#E8F5E9',
    'Saturating':   '#FFF3E0',
    'Accelerating': '#E3F2FD',
    'S-shape':      '#F3E5F5',
    'Flat/Weak':    '#F5F5F5',
    'U-shape':      '#FFF9C4',
    'Arch-shape':   '#FFE0B2',
    'Complex':      '#FFEBEE',
    'Insufficient data': '#FFFFFF',
}


def draw_model_zone_grid(x_var, y_var, pair_label, version):
    grid_runs = select_grid_runs(pair_label)
    grid_domains = select_grid_domains(pair_label, grid_runs)
    n_rows = len(grid_runs)
    n_cols = len(grid_domains)
    if n_rows == 0 or n_cols == 0:
        print(f'No models with data for {pair_label} {version}')
        return

    col_limits = {
        domain_key: column_limits_for_pair(pair_label, domain_key, grid_runs, version)
        for domain_key, _, _ in grid_domains
    }

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(3.2 * n_cols, 2.3 * n_rows),
        constrained_layout=True,
        squeeze=False,
    )

    for row_i, run in enumerate(grid_runs):
        payloads = _ensure_payloads(run)

        for col_i, (domain_key, domain_label, _) in enumerate(grid_domains):
            ax = axes[row_i, col_i]
            payload = payloads[(pair_label, domain_key)]
            x_data, y_data = payload['prepared'][version]
            groups = payload['prepared'][f'{version}_groups']
            scatter_points(ax, x_data, y_data, domain_key, groups)

            bin_curve = equal_count_bin_medians(x_data, y_data)
            if len(bin_curve) >= 2:
                ax.plot(
                    bin_curve['x'], bin_curve['y'],
                    color='white', linewidth=3.6, zorder=3,
                )
                ax.plot(
                    bin_curve['x'], bin_curve['y'],
                    color='#20252B', linewidth=1.7, marker='o',
                    markersize=2.8, markerfacecolor='white',
                    markeredgecolor='#20252B', markeredgewidth=0.7,
                    zorder=4,
                )
            draw_lowess_fit(ax, x_data, y_data)

            # LOWESS shape classification
            feat = extract_lowess_shape(x_data, y_data)
            shape_label = classify_lowess_shape(feat)

            # Build annotation with power-law metrics
            version_rows = payload['result_rows'].loc[
                payload['result_rows']['outlier_version'] == version
            ]
            if len(version_rows) == 1:
                result_row = version_rows.iloc[0]
                line1 = (
                    f"dcor={metric_text(result_row, 'dcor')}  "
                    f"r={metric_text(result_row, 'pearson_r_descriptive')}"
                )
            else:
                line1 = ''
            if feat is not None and np.isfinite(feat.get('power_b', np.nan)):
                line2 = (
                    f"b={feat['power_b']:.2f}  "
                    f"R²={feat['power_r2']:.3f}  "
                    f"|b-1|={feat['abs_b_minus_1']:.2f}"
                )
            else:
                line2 = ''
            line3 = f"n={len(x_data)}"
            parts = [p for p in [line1, line2, line3] if p]
            details = '\n'.join(parts)
            draw_class_annotation(ax, shape_label, details, fontsize=8.4)

            xlim, ylim = col_limits[domain_key]
            ax.set_xlim(xlim)
            ax.set_ylim(ylim)
            style_axes(ax)
            ax.set_facecolor(SHAPE_FACECOLORS.get(shape_label, '#FFFFFF'))

            if row_i == 0:
                ax.set_title(
                    f'{domain_label}\n({domain_key})',
                    fontsize=12, fontweight='semibold', pad=6,
                )
            if col_i == 0:
                ax.set_ylabel(UNIT_LABELS[y_var], fontsize=9)
                ax.text(
                    -0.42, 0.5, run['model'],
                    transform=ax.transAxes, rotation=90,
                    va='center', ha='center',
                    fontsize=12, fontweight='bold', color='#222222',
                )
            if row_i == n_rows - 1:
                ax.set_xlabel(UNIT_LABELS[x_var], fontsize=9)

    fig.suptitle(
        f'{version.upper()}  ·  {PAIR_FULL_NAMES.get(pair_label, pair_label)}  ·  rows = models, columns = zones',
        fontsize=17, fontweight='semibold',
    )
    legend_handles = [
        Line2D(
            [0], [0], color='#20252B', linewidth=1.7, marker='o',
            markersize=4, markerfacecolor='white', alpha=0.6,
            label='Median trend',
        ),
        Line2D(
            [0], [0], color='#B12A68', linewidth=1.6,
            linestyle='-', alpha=0.6,
            label='LOWESS fit',
        ),
    ]
    for key, label in ZONE_LEGEND:
        legend_handles.append(
            Line2D(
                [0], [0], marker='o', color='none',
                markerfacecolor=ZONE_COLORS[key], markeredgecolor='none',
                markersize=8, alpha=0.6, label=label,
            )
        )

    fig.legend(
        handles=legend_handles, loc='lower center',
        ncol=min(8, len(legend_handles)),
        bbox_to_anchor=(0.5, -0.06), frameon=True, fancybox=True, framealpha=0.5, edgecolor='none', fontsize=11,
    )
    if WRITE_OUTPUTS:
        path = FIGURE_DIR / (
            f'S4_modelgrid_{PAIR_SLUGS[pair_label]}_{version}.png'
        )
        fig.savefig(path, dpi=160, bbox_inches='tight')
        print(f'saved {path}')
    plt.show()
    plt.close(fig)


print('Model × zone grids — RAW')
print('=' * 72)
for x_var, y_var, pair_label in VARIABLE_PAIRS:
    print(f'\nRAW grid · {pair_label}')
    draw_model_zone_grid(x_var, y_var, pair_label, 'raw')


## 6f. Model × zone grid — TRIMMED / 模型×分区总图（trimmed）

Same layout as 6e, trimmed version.

**中文说明：** 与6e相同布局的trimmed版本。

In [9]:
# print('Model × zone grids — TRIMMED')
# print('=' * 72)
# for x_var, y_var, pair_label in VARIABLE_PAIRS:
#     print(f'\nTRIMMED grid · {pair_label}')
#     draw_model_zone_grid(x_var, y_var, pair_label, 'trimmed')


## 6g. Model × zone grid — GLOBAL TRIMMED / 模型×分区总图（全局去极值）

Same layout as 6f, using global (all_land) quantile bounds.

**中文说明：** 与6f相同布局，使用全局分位数边界。

In [ ]:
# def build_global_trim_bounds(run):
#     """Compute 1-99% quantile bounds from all_land for each variable pair."""
#     t = run['clim']
#     bounds = {}
#     for x_var, y_var, pair_label in VARIABLE_PAIRS:
#         if x_var not in t.columns or y_var not in t.columns:
#             continue
#         x_all = np.asarray(t[x_var], dtype=float)
#         y_all = np.asarray(t[y_var], dtype=float)
#         finite = np.isfinite(x_all) & np.isfinite(y_all)
#         xf, yf = x_all[finite], y_all[finite]
#         if len(xf) == 0:
#             bounds[(x_var, y_var)] = (np.nan, np.nan, np.nan, np.nan)
#         else:
#             x_lo, x_hi = np.quantile(xf, [TRIM_LOWER_QUANTILE, TRIM_UPPER_QUANTILE])
#             y_lo, y_hi = np.quantile(yf, [TRIM_LOWER_QUANTILE, TRIM_UPPER_QUANTILE])
#             bounds[(x_var, y_var)] = (float(x_lo), float(x_hi), float(y_lo), float(y_hi))
#     return bounds


# def build_panel_payloads_global(run):
#     """Like build_panel_payloads but uses all_land trim bounds for every domain."""
#     t = run['clim']
#     global_bounds = build_global_trim_bounds(run)
#     payloads = {}
#     for x_var, y_var, pair_label in VARIABLE_PAIRS:
#         if (x_var, y_var) not in global_bounds:
#             continue
#         gb = global_bounds[(x_var, y_var)]
#         x_lo, x_hi, y_lo, y_hi = gb
#         for domain_key, domain_label, domain_filter in DOMAINS:
#             subset = t.loc[domain_filter(t)] if domain_filter is not None else t
#             x_arr = np.asarray(subset[x_var], dtype=float)
#             y_arr = np.asarray(subset[y_var], dtype=float)
#             groups = np.asarray(subset['analysis_zone']) if 'analysis_zone' in subset else None
#             finite = np.isfinite(x_arr) & np.isfinite(y_arr)
#             x_raw = x_arr[finite]
#             y_raw = y_arr[finite]
#             g_raw = groups[finite] if groups is not None else None
#             if len(x_raw) == 0 or np.isnan(x_lo):
#                 keep = np.zeros(len(x_raw), dtype=bool)
#             else:
#                 keep = (
#                     (x_raw >= x_lo) & (x_raw <= x_hi)
#                     & (y_raw >= y_lo) & (y_raw <= y_hi)
#                 )
#             x_trim = x_raw[keep]
#             y_trim = y_raw[keep]
#             g_trim = g_raw[keep] if g_raw is not None else None
#             prepared = {
#                 'raw': (x_raw, y_raw),
#                 'trimmed': (x_trim, y_trim),
#                 'raw_groups': g_raw,
#                 'trimmed_groups': g_trim,
#                 'trim_keep': keep,
#                 'bounds': gb,
#             }
#             result_rows = RESULTS_DF.loc[
#                 (RESULTS_DF['model'] == run['model'])
#                 & (RESULTS_DF['experiment'] == run['experiment'])
#                 & (RESULTS_DF['member'] == run['member'])
#                 & (RESULTS_DF['grid'] == run['grid'])
#                 & (RESULTS_DF['domain'] == domain_key)
#                 & (RESULTS_DF['sample'] == 'all')
#                 & (RESULTS_DF['x_var'] == x_var)
#                 & (RESULTS_DF['y_var'] == y_var)
#             ]
#             payloads[(pair_label, domain_key)] = {
#                 'x_var': x_var,
#                 'y_var': y_var,
#                 'pair_label': pair_label,
#                 'domain_key': domain_key,
#                 'domain_label': domain_label,
#                 'prepared': prepared,
#                 'result_rows': result_rows,
#                 'xlim': padded_limits(x_trim),
#                 'ylim': padded_limits(y_trim),
#             }
#     return payloads


# def draw_model_zone_grid_global(x_var, y_var, pair_label):
#     """Model x zone grid using global trim bounds, showing only kept points."""
#     grid_runs = select_grid_runs(pair_label)
#     grid_domains = select_grid_domains(pair_label, grid_runs)
#     n_rows = len(grid_runs)
#     n_cols = len(grid_domains)
#     if n_rows == 0 or n_cols == 0:
#         print(f'No models with data for {pair_label} global-trimmed')
#         return

#     all_payloads = {}
#     for run in grid_runs:
#         all_payloads[run['model']] = build_panel_payloads_global(run)

#     col_limits = {}
#     for domain_key, _, _ in grid_domains:
#         xs, ys = [], []
#         for run in grid_runs:
#             p = all_payloads[run['model']].get((pair_label, domain_key))
#             if p is None:
#                 continue
#             xt, yt = p['prepared']['trimmed']
#             if len(xt) > 0:
#                 xs.append(xt)
#                 ys.append(yt)
#         if xs:
#             col_limits[domain_key] = (
#                 padded_limits(np.concatenate(xs)),
#                 padded_limits(np.concatenate(ys)),
#             )
#         else:
#             col_limits[domain_key] = ((-1.0, 1.0), (-1.0, 1.0))

#     fig, axes = plt.subplots(
#         n_rows, n_cols,
#         figsize=(3.2 * n_cols, 2.3 * n_rows),
#         constrained_layout=True,
#         squeeze=False,
#     )

#     for row_i, run in enumerate(grid_runs):
#         payloads = all_payloads[run['model']]
#         for col_i, (domain_key, domain_label, _) in enumerate(grid_domains):
#             ax = axes[row_i, col_i]
#             payload = payloads.get((pair_label, domain_key))
#             if payload is None:
#                 ax.text(
#                     0.5, 0.5, 'Unavailable', transform=ax.transAxes,
#                     ha='center', va='center', fontsize=12, color='#6B7280',
#                 )
#                 style_axes(ax)
#                 continue
#             x_data, y_data = payload['prepared']['trimmed']
#             groups = payload['prepared']['trimmed_groups']
#             scatter_points(ax, x_data, y_data, domain_key, groups)

#             bin_curve = equal_count_bin_medians(x_data, y_data)
#             if len(bin_curve) >= 2:
#                 ax.plot(
#                     bin_curve['x'], bin_curve['y'],
#                     color='white', linewidth=3.6, zorder=3,
#                 )
#                 ax.plot(
#                     bin_curve['x'], bin_curve['y'],
#                     color='#20252B', linewidth=1.7, marker='o',
#                     markersize=2.8, markerfacecolor='white',
#                     markeredgecolor='#20252B', markeredgewidth=0.7,
#                     zorder=4,
#                 )
#             draw_lowess_fit(ax, x_data, y_data)

#             version_rows = payload['result_rows'].loc[
#                 payload['result_rows']['outlier_version'] == 'trimmed'
#             ]
#             raw_rows = payload['result_rows'].loc[
#                 payload['result_rows']['outlier_version'] == 'raw'
#             ]
#             if len(version_rows) == 1:
#                 result_row = version_rows.iloc[0]
#                 raw_row = raw_rows.iloc[0] if len(raw_rows) == 1 else None
#                 class_name, details = format_panel_annotation(
#                     result_row, 'trimmed', raw_row=raw_row
#                 )
#             else:
#                 class_name, details = 'No result', ''
#             draw_class_annotation(ax, class_name, details, fontsize=8.4)

#             xlim, ylim = col_limits[domain_key]
#             ax.set_xlim(xlim)
#             ax.set_ylim(ylim)
#             style_axes(ax)
#             face = '#FFFFFF'
#             if len(version_rows) == 1:
#                 face = classification_facecolor(version_rows.iloc[0]['group'])
#             ax.set_facecolor(face)

#             if row_i == 0:
#                 ax.set_title(
#                     f'{domain_label}\n({domain_key})',
#                     fontsize=12, fontweight='semibold', pad=6,
#                 )
#             if col_i == 0:
#                 ax.set_ylabel(UNIT_LABELS[y_var], fontsize=9)
#                 ax.text(
#                     -0.42, 0.5, run['model'],
#                     transform=ax.transAxes, rotation=90,
#                     va='center', ha='center',
#                     fontsize=12, fontweight='bold', color='#222222',
#                 )
#             if row_i == n_rows - 1:
#                 ax.set_xlabel(UNIT_LABELS[x_var], fontsize=9)

#     fig.suptitle(
#         f'GLOBAL TRIMMED  ·  {PAIR_FULL_NAMES.get(pair_label, pair_label)}  ·  rows = models, columns = zones',
#         fontsize=17, fontweight='semibold',
#     )
#     legend_handles = [
#         Line2D(
#             [0], [0], color='#20252B', linewidth=1.7, marker='o',
#             markersize=4, markerfacecolor='white', alpha=0.6,
#             label='Median trend',
#         ),
#         Line2D(
#             [0], [0], color='#B12A68', linewidth=1.6,
#             linestyle='-', alpha=0.6,
#             label='LOWESS fit',
#         ),
#     ]
#     for key, label in ZONE_LEGEND:
#         legend_handles.append(
#             Line2D(
#                 [0], [0], marker='o', color='none',
#                 markerfacecolor=ZONE_COLORS[key], markeredgecolor='none',
#                 markersize=8, alpha=0.6, label=label,
#             )
#         )
#     fig.legend(
#         handles=legend_handles, loc='lower center',
#         ncol=min(7, len(legend_handles)),
#         bbox_to_anchor=(0.5, -0.06), frameon=True, fancybox=True, framealpha=0.5, edgecolor='none', fontsize=11,
#     )
#     if WRITE_OUTPUTS:
#         path = FIGURE_DIR / (
#             f'S4_modelgrid_{PAIR_SLUGS[pair_label]}_global_trimmed.png'
#         )
#         fig.savefig(path, dpi=160, bbox_inches='tight')
#         print(f'saved {path}')
#     plt.show()
#     plt.close(fig)


# print('Model × zone grids — GLOBAL TRIMMED')
# print('=' * 72)
# for x_var, y_var, pair_label in VARIABLE_PAIRS:
#     print(f'\nGLOBAL TRIMMED grid · {pair_label}')
#     draw_model_zone_grid_global(x_var, y_var, pair_label)


## 7. Classification heatmap — RAW / raw分类热力图

One heatmap per model for **raw** classifications (domain × variable pair). Trimmed heatmaps follow in the next section.

**中文说明：** 每个模型先画一张raw热力图；全部raw热力图完成后再画trimmed。


In [11]:
# [COMMENTED OUT — not needed for variable screening]
# To restore, remove the leading "# " from each line below.
# domain_order = [d[0] for d in DOMAINS]
# pair_order = [v[2] for v in VARIABLE_PAIRS]
# GROUP_COLORS = {
#     'Linear': '#4CAF50',
#     'Saturation': '#FF9800',
#     'Acceleration': '#F44336',
#     'U-shape': '#9C27B0',
#     'Cubic': '#3F51B5',
#     'Oscillation': '#00BCD4',
#     'Branch': '#795548',
#     'Transition': '#607D8B',
#     'No Global Relationship': '#BDBDBD',
#     'Uncertain': '#E0E0E0',
#     'Insufficient data': '#F3F4F6',
# }
# 
# 
# def draw_classification_heatmap(ax, result_table, version):
#     version_table = result_table.loc[
#         (result_table['sample'] == 'all')
#         & (result_table['outlier_version'] == version)
#     ]
#     pivot_group = version_table.pivot_table(
#         index='domain', columns='pair', values='group', aggfunc='first',
#     ).reindex(index=domain_order, columns=pair_order)
#     pivot_mic = version_table.pivot_table(
#         index='domain', columns='pair', values='mic', aggfunc='first',
#     ).reindex(index=domain_order, columns=pair_order)
# 
#     ax.set_xlim(-0.5, len(pair_order) - 0.5)
#     ax.set_ylim(-0.5, len(domain_order) - 0.5)
#     ax.invert_yaxis()
#     ax.set_xticks(range(len(pair_order)))
#     ax.set_xticklabels(pair_order)
#     ax.set_yticks(range(len(domain_order)))
#     ax.set_yticklabels([d[1] for d in DOMAINS])
#     for yi, domain in enumerate(domain_order):
#         for xi, pair in enumerate(pair_order):
#             group_value = pivot_group.loc[domain, pair]
#             group = group_value if pd.notna(group_value) else ''
#             mic_val = pivot_mic.loc[domain, pair]
#             rect = plt.Rectangle(
#                 (xi - 0.48, yi - 0.45), 0.96, 0.9,
#                 facecolor=GROUP_COLORS.get(group, '#F5F5F5'),
#                 edgecolor='white', linewidth=2,
#             )
#             ax.add_patch(rect)
#             ax.text(xi, yi - 0.1, group, ha='center', va='center',
#                     fontsize=11, fontweight='semibold', color='#222222')
#             if pd.notna(mic_val):
#                 ax.text(xi, yi + 0.2, f'MIC={mic_val:.3f}',
#                         ha='center', va='center', fontsize=10, color='#555555')
#     ax.set_title(f'{version.upper()} classification',
#                  fontsize=14, fontweight='semibold', pad=10)
#     style_axes(ax)
#     ax.grid(False)
# 
# 
# def draw_heatmap_figure(run, version):
#     run_results = RESULTS_DF.loc[
#         (RESULTS_DF['model'] == run['model'])
#         & (RESULTS_DF['experiment'] == run['experiment'])
#         & (RESULTS_DF['member'] == run['member'])
#         & (RESULTS_DF['grid'] == run['grid'])
#     ]
#     fig, ax = plt.subplots(figsize=(10.5, 6.8), constrained_layout=True)
#     draw_classification_heatmap(ax, run_results, version)
#     fig.suptitle(
#         f"{version.upper()} spatial classifications · {run['model']}",
#         fontsize=14, fontweight='semibold',
#     )
#     if WRITE_OUTPUTS:
#         fig.savefig(
#             run['zones_dir'] / f'S4_classification_heatmap_{version}.png',
#             bbox_inches='tight',
#         )
#     plt.show()
#     plt.close(fig)
# 
# 
# print('RAW classification heatmaps')
# print('=' * 72)
# for run in RUNS:
#     draw_heatmap_figure(run, 'raw')
# 

## 7b. Classification heatmap — TRIMMED / trimmed分类热力图

Same domain × pair layout as the raw heatmaps, shown only after every model's raw heatmap.

**中文说明：** 与raw热力图布局相同，全部raw热力图完成后再显示。


In [12]:
# [COMMENTED OUT — not needed for variable screening]
# To restore, remove the leading "# " from each line below.
# print('TRIMMED classification heatmaps')
# print('=' * 72)
# for run in RUNS:
#     draw_heatmap_figure(run, 'trimmed')
# 

## 8. Sensitivity analysis / 敏感性分析

Two independent sensitivity checks are reported. First, all versus core is compared separately for raw and trimmed data to diagnose boundary mixing. Second, raw versus trimmed is compared for every domain × sample × pair to diagnose dependence on extreme grid cells. Raw remains the primary result.

**中文说明：** 这里分开检查两种敏感性：(1) 在raw和trimmed内部各自比较all与core，用于判断分区边界混合；(2) 对每个domain × sample × pair比较raw与trimmed，用于判断分类是否依赖极端格点。raw始终是主结果。


In [13]:
# [COMMENTED OUT — not needed for variable screening]
# To restore, remove the leading "# " from each line below.
# RUN_KEYS = ['model', 'experiment', 'member', 'grid']
# zone_domains = ['WW', 'WD', 'CW', 'CD']
# 
# # A. all versus core, evaluated separately inside raw and trimmed.
# core_sensitivity_rows = []
# for run_key, run_cases in RESULTS_DF.groupby(RUN_KEYS, sort=False):
#     run_meta = dict(zip(RUN_KEYS, run_key))
#     for version in ['raw', 'trimmed']:
#         version_cases = run_cases.loc[run_cases['outlier_version'] == version]
#         for domain in zone_domains:
#             for x_var, y_var, pair_label in VARIABLE_PAIRS:
#                 common = (
#                     (version_cases['domain'] == domain)
#                     & (version_cases['x_var'] == x_var)
#                     & (version_cases['y_var'] == y_var)
#                 )
#                 all_row = version_cases.loc[common & version_cases['sample'].eq('all')]
#                 core_row = version_cases.loc[common & version_cases['sample'].eq('core')]
#                 if len(all_row) != 1 or len(core_row) != 1:
#                     continue
#                 a = all_row.iloc[0]
#                 c = core_row.iloc[0]
#                 core_sensitivity_rows.append({
#                     **run_meta,
#                     'outlier_version': version,
#                     'domain': domain,
#                     'pair': pair_label,
#                     'all_n': a['n_cells'],
#                     'all_group': a['group'],
#                     'all_mic': a['mic'],
#                     'core_n': c['n_cells'],
#                     'core_group': c['group'],
#                     'core_mic': c['mic'],
#                     'consistent': a['group'] == c['group'],
#                 })
# 
# core_sensitivity_df = pd.DataFrame(core_sensitivity_rows)
# print('\n=== A. All vs core sensitivity ===')
# for _, _, pair_label in VARIABLE_PAIRS:
#     print(f'\n--- {pair_label} ---')
#     display(core_sensitivity_df.loc[
#         core_sensitivity_df['pair'] == pair_label,
#         ['domain', 'outlier_version', 'all_n', 'all_group', 'all_mic',
#          'core_n', 'core_group', 'core_mic', 'consistent'],
#     ].round(4))
# 
# # B. raw versus trimmed for every exact domain × sample × pair.
# outlier_sensitivity_rows = []
# unit_cols = RUN_KEYS + ['domain', 'sample', 'x_var', 'y_var', 'pair']
# for unit_key, unit_cases in RESULTS_DF.groupby(unit_cols, sort=False):
#     unit_meta = dict(zip(unit_cols, unit_key))
#     raw_row = unit_cases.loc[unit_cases['outlier_version'] == 'raw']
#     trimmed_row = unit_cases.loc[unit_cases['outlier_version'] == 'trimmed']
#     if len(raw_row) != 1 or len(trimmed_row) != 1:
#         continue
#     raw = raw_row.iloc[0]
#     trimmed = trimmed_row.iloc[0]
#     outlier_sensitivity_rows.append({
#         **unit_meta,
#         'raw_n': raw['n_cells'],
#         'trimmed_n': trimmed['n_cells'],
#         'n_removed': trimmed['n_removed'],
#         'removed_fraction': trimmed['removed_fraction'],
#         'raw_mic': raw.get('mic', np.nan),
#         'trimmed_mic': trimmed.get('mic', np.nan),
#         'raw_pearson': raw.get('pearson_r_descriptive', np.nan),
#         'trimmed_pearson': trimmed.get('pearson_r_descriptive', np.nan),
#         'raw_power_r2': raw.get('power_r2', np.nan),
#         'trimmed_power_r2': trimmed.get('power_r2', np.nan),
#         'raw_power_b': raw.get('power_b', np.nan),
#         'trimmed_power_b': trimmed.get('power_b', np.nan),
#         'raw_group': raw['group'],
#         'trimmed_group': trimmed['group'],
#         'consistent': raw['group'] == trimmed['group'],
#     })
# 
# outlier_sensitivity_df = pd.DataFrame(outlier_sensitivity_rows)
# print('\n=== B. Raw vs trimmed outlier sensitivity ===')
# outlier_display_cols = [
#     'domain', 'sample', 'raw_n', 'trimmed_n', 'n_removed',
#     'removed_fraction', 'raw_mic', 'trimmed_mic',
#     'raw_pearson', 'trimmed_pearson',
#     'raw_power_r2', 'trimmed_power_r2',
#     'raw_power_b', 'trimmed_power_b',
#     'raw_group', 'trimmed_group', 'consistent',
# ]
# for _, _, pair_label in VARIABLE_PAIRS:
#     print(f'\n--- {pair_label} ---')
#     display(outlier_sensitivity_df.loc[
#         outlier_sensitivity_df['pair'] == pair_label,
#         outlier_display_cols,
#     ].round(4))
# 
# n_changed = int((~outlier_sensitivity_df['consistent']).sum())
# print(f'\nRaw/trimmed classification changes: {n_changed} / {len(outlier_sensitivity_df)}')
# if n_changed > 0:
#     display(outlier_sensitivity_df.loc[
#         ~outlier_sensitivity_df['consistent'],
#         ['domain', 'sample', 'pair', 'raw_group', 'trimmed_group',
#          'removed_fraction'],
#     ].round(4))

## 6i. LOWESS shape classification / LOWESS 形态分类

> Disabled. Unused for the current MIC/dCor gate → branch workflow.

**中文说明：** 已注释。当前主流程不跑这一节。


In [14]:
%%script --no-raise-error false
# Disabled: unused for the MIC/dCor gate → branch workflow (sections 14–15).
from scipy.interpolate import interp1d


def extract_lowess_shape(x_data, y_data, frac=0.3, it=3):
    """Fit LOWESS and extract shape features from the smoothed curve."""
    from scipy.signal import find_peaks

    x_arr = np.asarray(x_data, dtype=float)
    y_arr = np.asarray(y_data, dtype=float)
    finite = np.isfinite(x_arr) & np.isfinite(y_arr)
    xf, yf = x_arr[finite], y_arr[finite]
    if len(xf) < 20:
        return None

    result = _lowess(yf, xf, frac=frac, it=it, return_sorted=True)
    xs, ys = result[:, 0], result[:, 1]

    y_range = ys.max() - ys.min()
    x_range = xs.max() - xs.min()
    if x_range == 0 or y_range == 0:
        return None

    # Power-law fit on LOWESS curve: y = a * x^b
    # Fit in log-log space, evaluate R² in original space
    power_b = np.nan
    power_r2 = np.nan
    abs_b_minus_1 = np.nan
    pos_mask = (xs > 0) & (ys > 0)
    if pos_mask.sum() >= 10:
        log_x = np.log(xs[pos_mask])
        log_y = np.log(ys[pos_mask])
        coef = np.polyfit(log_x, log_y, 1)
        power_b = coef[0]
        abs_b_minus_1 = abs(power_b - 1.0)
        # R² in original space: power law prediction vs LOWESS curve
        y_pred = np.exp(np.polyval(coef, log_x))
        y_actual = ys[pos_mask]
        ss_res = np.sum((y_actual - y_pred) ** 2)
        ss_tot = np.sum((y_actual - y_actual.mean()) ** 2)
        power_r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else 1.0

    # First derivative
    dx = np.diff(xs)
    dy = np.diff(ys)
    mask = dx > 0
    dx, dy = dx[mask], dy[mask]
    if len(dx) < 5:
        return None
    slope = dy / dx

    # Second derivative
    x_mid = 0.5 * (xs[:-1][mask] + xs[1:][mask])
    dx2 = np.diff(x_mid)
    dslope = np.diff(slope)
    mask2 = dx2 > 0
    curvature = dslope[mask2] / dx2[mask2] if mask2.any() else np.array([])

    norm_slope = slope * (x_range / y_range)

    frac_positive = (slope > 0).mean()
    frac_negative = (slope < 0).mean()
    mean_norm_slope = norm_slope.mean()
    slope_std = norm_slope.std()

    n_edge = max(3, int(0.2 * len(slope)))
    slope_start = float(np.median(slope[:n_edge]))
    slope_end = float(np.median(slope[-n_edge:]))

    if len(curvature) > 0:
        frac_concave = (curvature > 0).mean()
        mean_curvature = float(np.mean(curvature * (x_range**2 / y_range)))
        half = len(curvature) // 2
        curv_first_half = curvature[:half].mean() if half > 0 else 0
        curv_second_half = curvature[half:].mean()
    else:
        frac_concave = np.nan
        mean_curvature = np.nan
        curv_first_half = 0
        curv_second_half = 0

    net_change = (ys[-1] - ys[0]) / y_range if y_range > 0 else 0

    # Turning points via find_peaks with prominence
    amplitude = np.ptp(ys)
    n_peaks, n_troughs = 0, 0
    if amplitude > 1e-10:
        prom = max(0.10 * amplitude, 1e-10)
        min_dist = max(5, int(0.08 * len(ys)))
        peaks, _ = find_peaks(ys, prominence=prom, distance=min_dist)
        troughs, _ = find_peaks(-ys, prominence=prom, distance=min_dist)
        n_peaks = len(peaks)
        n_troughs = len(troughs)

    return {
        'frac_positive_slope': frac_positive,
        'frac_negative_slope': frac_negative,
        'mean_norm_slope': mean_norm_slope,
        'slope_std': slope_std,
        'slope_start': slope_start,
        'slope_end': slope_end,
        'frac_concave': frac_concave,
        'mean_curvature': mean_curvature,
        'net_change': net_change,
        'curv_first_half': curv_first_half,
        'curv_second_half': curv_second_half,
        'power_b': power_b,
        'power_r2': power_r2,
        'abs_b_minus_1': abs_b_minus_1,
        'n_peaks': n_peaks,
        'n_troughs': n_troughs,
    }


def classify_lowess_shape(feat, flat_thresh=0.15):
    """Assign a shape label based on extracted LOWESS features."""
    if feat is None:
        return 'Insufficient data'

    net = feat['net_change']
    frac_pos = feat['frac_positive_slope']
    frac_neg = feat['frac_negative_slope']
    frac_concave = feat['frac_concave']
    power_b = feat['power_b']
    power_r2 = feat['power_r2']
    abs_b_1 = feat['abs_b_minus_1']
    n_peaks = feat['n_peaks']
    n_troughs = feat['n_troughs']
    n_tp = n_peaks + n_troughs

    # 1. Flat / Weak
    if abs(net) < flat_thresh:
        if n_peaks == 1 and n_troughs == 0:
            return 'Arch-shape'
        if n_troughs == 1 and n_peaks == 0:
            return 'U-shape'
        return 'Flat/Weak'

    # 2. Power-law fit first — if it fits well, use b to classify
    has_power = np.isfinite(power_b) and np.isfinite(power_r2)
    is_monotone_inc = frac_pos >= 0.80
    is_monotone_dec = frac_neg >= 0.80
    is_monotone = is_monotone_inc or is_monotone_dec

    if has_power and power_r2 >= 0.80 and is_monotone:
        if abs_b_1 < 0.15:
            return 'Linear'
        if power_b > 1.15:
            return 'Accelerating'
        if power_b < 0.85:
            # Cross-check: if curvature says concave-up, trust curvature
            if is_monotone_inc and frac_concave > 0.55:
                return 'Accelerating'
            return 'Saturating'

    # 3. Power-law fit poor — check turning points
    if n_tp >= 3:
        return 'Complex'
    if n_peaks == 1 and n_troughs == 0:
        return 'Arch-shape'
    if n_troughs == 1 and n_peaks == 0:
        return 'U-shape'
    if n_tp == 2:
        return 'S-shape'

    # 4. Monotone but power-law R² low — use curvature only if R² is moderate
    if is_monotone:
        if not has_power or power_r2 < 0.50:
            return 'Complex'
        if is_monotone_inc and frac_concave < 0.40:
            return 'Saturating'
        if is_monotone_dec and frac_concave > 0.60:
            return 'Saturating'
        if is_monotone_inc and frac_concave > 0.60:
            return 'Accelerating'
        if is_monotone_dec and frac_concave < 0.40:
            return 'Accelerating'
        return 'Linear'

    # 5. Non-monotone without clear turning points
    c1 = feat['curv_first_half']
    c2 = feat['curv_second_half']
    if c1 * c2 < 0:
        return 'S-shape'

    return 'Complex' 
# --- Run shape extraction on all model x zone x pair x version combos ---
shape_records = []
for run in RUNS:
    payloads = _ensure_payloads(run)
    for x_var, y_var, pair_label in VARIABLE_PAIRS:
        for domain_key, domain_label, _ in DOMAINS:
            payload = payloads.get((pair_label, domain_key))
            if payload is None:
                continue
            for version in ['raw', 'trimmed']:
                x_data, y_data = payload['prepared'][version]
                if len(x_data) < 20:
                    shape_records.append({
                        'model': run['model'], 'domain': domain_key,
                        'x_var': x_var, 'y_var': y_var,
                        'pair': pair_label, 'version': version,
                        'shape_label': 'Insufficient data', 'n_points': len(x_data),
                    })
                    continue
                feat = extract_lowess_shape(x_data, y_data)
                label = classify_lowess_shape(feat)
                rec = {
                    'model': run['model'], 'domain': domain_key,
                    'x_var': x_var, 'y_var': y_var,
                    'pair': pair_label, 'version': version,
                    'shape_label': label,
                    'n_points': len(x_data),
                }
                if feat is not None:
                    rec.update(feat)
                shape_records.append(rec)

LOWESS_SHAPE_DF = pd.DataFrame(shape_records)
print(f'LOWESS shape records: {len(LOWESS_SHAPE_DF)}')
print()
print('Shape label counts (all versions):')
print(LOWESS_SHAPE_DF['shape_label'].value_counts())
print()

for version in ['raw', 'trimmed']:
    print(f'\n=== {version.upper()} ===')
    sub = LOWESS_SHAPE_DF[LOWESS_SHAPE_DF['version'] == version]
    for pair_label in [p[2] for p in VARIABLE_PAIRS]:
        ps = sub[sub['pair'] == pair_label]
        print(f'\n{pair_label}:')
        ct = ps.pivot_table(
            index='model', columns='domain',
            values='shape_label', aggfunc='first',
        )
        col_order = [d[0] for d in DOMAINS if d[0] in ct.columns]
        print(ct[col_order].to_string())

if WRITE_OUTPUTS:
    shape_path = S4_OUTPUT_DIR / f'S4_lowess_shape_{START_YEAR}_{END_YEAR}.csv'
    LOWESS_SHAPE_DF.to_csv(shape_path, index=False)
    print(f'\nSaved: {shape_path}')


## 6h. Density plots — KDE contour & 2D histogram / 密度图

> Disabled. Unused for the current MIC/dCor gate → branch workflow.

**中文说明：** 已注释。当前主流程不跑这一节。


In [15]:
%%script --no-raise-error false
# Disabled: unused for the MIC/dCor gate → branch workflow (sections 14–15).
def draw_density_grid(x_var, y_var, pair_label, version, method='kde'):
    """Model × zone density grid. version: 'raw', 'trimmed', or 'global_trimmed'."""
    method_label = 'KDE' if method == 'kde' else 'Hexbin'
    version_label = version.upper().replace('_', ' ')

    if version == 'global_trimmed':
        grid_runs = select_grid_runs(pair_label)
        grid_domains = select_grid_domains(pair_label, grid_runs)
        all_payloads = {}
        for run in grid_runs:
            all_payloads[run['model']] = build_panel_payloads_global(run)
    else:
        grid_runs = select_grid_runs(pair_label)
        grid_domains = select_grid_domains(pair_label, grid_runs)

    n_rows = len(grid_runs)
    n_cols = len(grid_domains)
    if n_rows == 0 or n_cols == 0:
        print(f'No models with data for {pair_label} {version_label}')
        return

    if version != 'global_trimmed':
        col_limits = {
            dk: column_limits_for_pair(pair_label, dk, grid_runs, version)
            for dk, _, _ in grid_domains
        }
    else:
        col_limits = {}
        for dk, _, _ in grid_domains:
            xs, ys = [], []
            for run in grid_runs:
                p = all_payloads[run['model']].get((pair_label, dk))
                if p is None:
                    continue
                xt, yt = p['prepared']['trimmed']
                if len(xt) > 0:
                    xs.append(xt)
                    ys.append(yt)
            if xs:
                col_limits[dk] = (padded_limits(np.concatenate(xs)),
                                  padded_limits(np.concatenate(ys)))
            else:
                col_limits[dk] = ((-1.0, 1.0), (-1.0, 1.0))

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(3.2 * n_cols, 2.3 * n_rows),
        constrained_layout=True, squeeze=False,
    )

    for row_i, run in enumerate(grid_runs):
        if version == 'global_trimmed':
            payloads = all_payloads[run['model']]
        else:
            payloads = _ensure_payloads(run)

        for col_i, (dk, dl, _) in enumerate(grid_domains):
            ax = axes[row_i, col_i]
            payload = payloads.get((pair_label, dk))
            if payload is None:
                ax.text(0.5, 0.5, 'Unavailable', transform=ax.transAxes,
                        ha='center', va='center', fontsize=12, color='#6B7280')
                style_axes(ax)
                continue

            v_key = 'trimmed' if version in ('trimmed', 'global_trimmed') else 'raw'
            x_data, y_data = payload['prepared'][v_key]

            if method == 'kde':
                draw_density_kde(ax, x_data, y_data)
            else:
                draw_density_hexbin(ax, x_data, y_data)

            bin_curve = equal_count_bin_medians(x_data, y_data)
            if len(bin_curve) >= 2:
                ax.plot(bin_curve['x'], bin_curve['y'],
                        color='white', linewidth=3.0, zorder=6)
                ax.plot(bin_curve['x'], bin_curve['y'],
                        color='#20252B', linewidth=1.5, marker='o',
                        markersize=2.5, markerfacecolor='white',
                        markeredgecolor='#20252B', markeredgewidth=0.6,
                        zorder=7)
            draw_lowess_fit(ax, x_data, y_data)

            version_rows = payload['result_rows'].loc[
                payload['result_rows']['outlier_version'] == v_key
            ]
            raw_rows = payload['result_rows'].loc[
                payload['result_rows']['outlier_version'] == 'raw'
            ]
            # LOWESS shape classification
            feat = extract_lowess_shape(x_data, y_data)
            shape_label = classify_lowess_shape(feat)

            if len(version_rows) == 1:
                result_row = version_rows.iloc[0]
                line1 = (
                    f"dcor={metric_text(result_row, 'dcor')}  "
                    f"r={metric_text(result_row, 'pearson_r_descriptive')}"
                )
            else:
                line1 = ''
            if feat is not None and np.isfinite(feat.get('power_b', np.nan)):
                line2 = (
                    f"b={feat['power_b']:.2f}  "
                    f"R\u00b2={feat['power_r2']:.3f}  "
                    f"|b-1|={feat['abs_b_minus_1']:.2f}"
                )
            else:
                line2 = ''
            line3 = f'n={len(x_data)}'
            parts = [p for p in [line1, line2, line3] if p]
            details = '\n'.join(parts)
            draw_class_annotation(ax, shape_label, details, fontsize=8.4)

            xlim, ylim = col_limits[dk]
            ax.set_xlim(xlim)
            ax.set_ylim(ylim)
            style_axes(ax)

            if row_i == 0:
                ax.set_title(f'{dl}\n({dk})',
                             fontsize=12, fontweight='semibold', pad=6)
            if col_i == 0:
                ax.set_ylabel(UNIT_LABELS.get(y_var, y_var), fontsize=9)
                ax.text(-0.42, 0.5, run['model'],
                        transform=ax.transAxes, rotation=90,
                        va='center', ha='center',
                        fontsize=12, fontweight='bold', color='#222222')
            if row_i == n_rows - 1:
                ax.set_xlabel(UNIT_LABELS.get(x_var, x_var), fontsize=9)

    fig.suptitle(
        f'{version_label} · {method_label}  ·  '
        f'{PAIR_FULL_NAMES.get(pair_label, pair_label)}  ·  rows = models, columns = zones',
        fontsize=17, fontweight='semibold',
    )
    legend_handles = [
        Line2D([0], [0], color='#20252B', linewidth=1.7, marker='o',
               markersize=4, markerfacecolor='white', alpha=0.6,
               label='Median trend'),
        Line2D([0], [0], color='#B12A68', linewidth=1.2,
               linestyle='-', alpha=0.7, label='LOWESS fit'),
    ]
    fig.legend(handles=legend_handles, loc='lower center',
               ncol=2, bbox_to_anchor=(0.5, -0.06),
               frameon=True, fancybox=True, framealpha=0.5,
               edgecolor='none', fontsize=11)
    if WRITE_OUTPUTS:
        slug = PAIR_SLUGS[pair_label]
        path = FIGURE_DIR / f'S4_density_{method}_{slug}_{version}.png'
        fig.savefig(path, dpi=160, bbox_inches='tight')
        print(f'saved {path}')
    plt.show()
    plt.close(fig)


# for method in ['kde', 'hexbin']:
for method in ['hexbin']:
    method_label = 'KDE contour' if method == 'kde' else 'Hexbin'
    for version in ['raw', 'trimmed']:
        print(f'\nDensity grids — {method_label} · {version.upper()}')
        print('=' * 72)
        for x_var, y_var, pair_label in VARIABLE_PAIRS:
            print(f'  {pair_label}')
            draw_density_grid(x_var, y_var, pair_label, version, method)


## 9. Save results / 保存结果

Save the MIC / dCor gate table as parquet plus per-model CSVs. Branch detector outputs are written in Sections 14–15.

**中文说明：** 这里保存 MIC / dCor gate 结果。Branch 分类结果在第 14–15 节另外保存。


In [ ]:
if WRITE_OUTPUTS:
    results_path = S4_OUTPUT_DIR / f'S4_classification_results_{START_YEAR}_{END_YEAR}.parquet'
    RESULTS_DF.to_parquet(results_path, index=False)
    print(f'Saved {results_path}  ({len(RESULTS_DF)} rows)')

    # Per-model CSV splits
    for run in RUNS:
        model = run['model']
        run_results = RESULTS_DF.loc[
            (RESULTS_DF['model'] == model)
            & (RESULTS_DF['experiment'] == run['experiment'])
            & (RESULTS_DF['member'] == run['member'])
            & (RESULTS_DF['grid'] == run['grid'])
        ]
        path = S4_OUTPUT_DIR / f'S4_classification_{model}_{START_YEAR}_{END_YEAR}.csv'
        run_results.to_csv(path, index=False)
        print(f'  {model}: {len(run_results)} rows → {path.name}')

    # Manifest
    all_outputs = sorted(S4_OUTPUT_DIR.glob('S4_*'))
    manifest = pd.DataFrame([{
        'product': p.name,
        'size_MB': p.stat().st_size / 1e6,
    } for p in all_outputs])
    display(manifest.round({'size_MB': 3}))

print('S4 association gates saved.')


## 10. Legacy multi-method branch benchmark — disabled

> The earlier Dip/KDE/GMM/mixture-regression/density-ridge/principal-tree benchmark is retained as commented code for provenance. It is not executed. The active calculation now uses only the multi-resolution Hexbin detector in Section 12.


In [17]:
# LEGACY BRANCH BENCHMARK DISABLED: replaced by the Hexbin-only section below.
# Retained as comments for provenance; this cell intentionally performs no work.
# import importlib
# import sys
#
# if str(CASE_DIR) not in sys.path:
#     sys.path.insert(0, str(CASE_DIR))
#
# import branch_benchmark as _branch_benchmark
# _branch_benchmark = importlib.reload(_branch_benchmark)
#
# from branch_benchmark import (
#     detect_current_dip,
#     detect_density_ridge,
#     detect_mixture_regression,
#     detect_modal_kde,
#     detect_principal_tree,
#     detect_sliding_gmm,
#     plot_branch_result,
#     run_all_branch_detectors,
# )
#
#
# def get_real_branch_case(model, domain, x_var='P', y_var='Q'):
#     run = next(item for item in RUNS if item['model'] == model)
#     frame = run['clim']
#     if domain == 'all_land':
#         subset = frame
#     else:
#         subset = frame.loc[frame['analysis_zone'] == domain]
#     columns = [x_var, y_var]
#     for optional in ['lat', 'lon', 'analysis_zone', 'land_ice_fraction']:
#         if optional in subset.columns and optional not in columns:
#             columns.append(optional)
#     subset = subset[columns].replace([np.inf, -np.inf], np.nan)
#     subset = subset.dropna(subset=[x_var, y_var]).copy()
#     return subset
#
#
# # The first case is the user-identified branch-like P→Q panel. The remaining
# # real panels are comparison cases, not asserted ground truth.
# REAL_BRANCH_CASE_SPECS = [
#     ('target_CMCC_LI_P_Q', 'CMCC-CM2-SR5', 'LI', 'P', 'Q'),
#     ('comparison_CNRM_LI_P_Q', 'CNRM-CM6-1', 'LI', 'P', 'Q'),
#     ('comparison_GFDL_LI_P_Q', 'GFDL-CM4', 'LI', 'P', 'Q'),
#     ('comparison_CMCC_WW_P_Q', 'CMCC-CM2-SR5', 'WW', 'P', 'Q'),
# ]
#
# REAL_BRANCH_CASES = {
#     name: get_real_branch_case(model, domain, x_var, y_var)
#     for name, model, domain, x_var, y_var in REAL_BRANCH_CASE_SPECS
# }
#
#
# def make_synthetic_branch_cases(seed=42, n=3000):
#     rng = np.random.default_rng(seed)
#     x = rng.uniform(0, 1, n)
#     selector = rng.random(n)
#     cases = {
#         'synthetic_single_linear': (
#             x, 2 * x + rng.normal(0, 0.12, n), False,
#         ),
#         'synthetic_single_fan': (
#             x, 2 * x + rng.normal(0, 0.05 + 0.50 * x, n), False,
#         ),
#         'synthetic_two_linear_branches': (
#             x,
#             np.where(selector < 0.35, 2 * x + 1, 2 * x)
#             + rng.normal(0, 0.08, n),
#             True,
#         ),
#         'synthetic_two_curved_branches': (
#             x,
#             np.where(selector < 0.35, np.sin(4 * x) + 1.2, np.sin(4 * x))
#             + rng.normal(0, 0.08, n),
#             True,
#         ),
#         'synthetic_zero_plus_positive_branch': (
#             x,
#             np.where(
#                 selector < 0.70,
#                 rng.normal(0, 0.015, n),
#                 1.4 * x + rng.normal(0, 0.10, n),
#             ),
#             True,
#         ),
#     }
#     return cases
#
#
# SYNTHETIC_BRANCH_CASES = make_synthetic_branch_cases()
# print(f'Real benchmark cases: {len(REAL_BRANCH_CASES)}')
# print(f'Synthetic benchmark cases: {len(SYNTHETIC_BRANCH_CASES)}')

In [18]:
# LEGACY BRANCH BENCHMARK DISABLED: replaced by the Hexbin-only section below.
# Retained as comments for provenance; this cell intentionally performs no work.
# import time as _branch_time
#
# _branch_rows = []
# TARGET_BRANCH_RESULTS = []
#
# for case_name, (x_values, y_values, expected) in SYNTHETIC_BRANCH_CASES.items():
#     started = _branch_time.perf_counter()
#     results = run_all_branch_detectors(x_values, y_values)
#     elapsed = _branch_time.perf_counter() - started
#     for result in results:
#         row = result.to_dict()
#         row.update({
#             'case': case_name,
#             'case_type': 'synthetic',
#             'expected_branch': bool(expected),
#             'n_points': len(x_values),
#             'suite_runtime_s': elapsed,
#         })
#         _branch_rows.append(row)
#
# for case_name, frame in REAL_BRANCH_CASES.items():
#     started = _branch_time.perf_counter()
#     results = run_all_branch_detectors(frame['P'], frame['Q'])
#     elapsed = _branch_time.perf_counter() - started
#     if case_name == 'target_CMCC_LI_P_Q':
#         TARGET_BRANCH_RESULTS = results
#     for result in results:
#         row = result.to_dict()
#         row.update({
#             'case': case_name,
#             'case_type': 'real',
#             'expected_branch': np.nan,
#             'n_points': len(frame),
#             'suite_runtime_s': elapsed,
#         })
#         _branch_rows.append(row)
#
# BRANCH_BENCHMARK_DF = pd.DataFrame(_branch_rows)
#
# synthetic_matrix = BRANCH_BENCHMARK_DF.loc[
#     BRANCH_BENCHMARK_DF['case_type'] == 'synthetic'
# ].pivot(index='case', columns='method', values='branch_detected')
#
# real_matrix = BRANCH_BENCHMARK_DF.loc[
#     BRANCH_BENCHMARK_DF['case_type'] == 'real'
# ].pivot(index='case', columns='method', values='branch_detected')
#
# print('Synthetic cases: detected branch (known truth in case name/table)')
# display(synthetic_matrix.astype(int))
# print('Real CMIP6 cases: method comparison (not treated as ground truth)')
# display(real_matrix.astype(int))
#
#
# def binary_method_scores(table):
#     records = []
#     for method, group in table.groupby('method'):
#         truth = group['expected_branch'].astype(bool).to_numpy()
#         prediction = group['branch_detected'].astype(bool).to_numpy()
#         tp = int(np.sum(truth & prediction))
#         fp = int(np.sum(~truth & prediction))
#         fn = int(np.sum(truth & ~prediction))
#         tn = int(np.sum(~truth & ~prediction))
#         precision = tp / (tp + fp) if tp + fp else 0.0
#         recall = tp / (tp + fn) if tp + fn else 0.0
#         f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
#         records.append({
#             'method': method, 'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
#             'precision': precision, 'recall': recall, 'F1': f1,
#         })
#     return pd.DataFrame(records).sort_values(['F1', 'precision'], ascending=False)
#
#
# BRANCH_SYNTHETIC_SCORES_DF = binary_method_scores(
#     BRANCH_BENCHMARK_DF.loc[BRANCH_BENCHMARK_DF['case_type'] == 'synthetic']
# )
# print('Known-truth synthetic benchmark scores')
# display(BRANCH_SYNTHETIC_SCORES_DF.round(3))

In [19]:
# LEGACY BRANCH BENCHMARK DISABLED: replaced by the Hexbin-only section below.
# Retained as comments for provenance; this cell intentionally performs no work.
# target_frame = REAL_BRANCH_CASES['target_CMCC_LI_P_Q']
# target_x = target_frame['P'].to_numpy()
# target_y = target_frame['Q'].to_numpy()
#
# _stability_runs = []
#
#
# def add_stability(method, setting, result):
#     _stability_runs.append({
#         'method': method,
#         'setting': str(setting),
#         'branch_detected': result.branch_detected,
#         'n_branches': result.n_branches,
#         'x_coverage': result.x_coverage,
#         'persistence': result.persistence,
#         'score': result.score,
#     })
#
#
# for threshold in [0.20, 0.30, 0.40]:
#     add_stability(
#         'Current conditional Dip', f'fraction_threshold={threshold:.2f}',
#         detect_current_dip(target_x, target_y, fraction_threshold=threshold),
#     )
# for fraction in [0.05, 0.06, 0.08, 0.10]:
#     add_stability(
#         'Sliding KDE modal', f'window_frac={fraction:.2f}',
#         detect_modal_kde(target_x, target_y, window_frac=fraction),
#     )
# for fraction in [0.05, 0.07, 0.09, 0.10]:
#     add_stability(
#         'Sliding-window GMM', f'window_frac={fraction:.2f}',
#         detect_sliding_gmm(target_x, target_y, window_frac=fraction),
#     )
# for degree in [1, 2, 3]:
#     add_stability(
#         'Mixture of regressions', f'polynomial_degree={degree}',
#         detect_mixture_regression(target_x, target_y, degree=degree),
#     )
# for factor in [0.60, 0.75, 0.90, 1.05]:
#     add_stability(
#         '2-D KDE density ridge', f'bandwidth_factor={factor:.2f}',
#         detect_density_ridge(target_x, target_y, bandwidth_factor=factor),
#     )
# for nodes in [20, 28, 36, 44]:
#     add_stability(
#         'Principal tree (MST)', f'n_nodes={nodes}',
#         detect_principal_tree(target_x, target_y, n_nodes=nodes),
#     )
#
# BRANCH_STABILITY_DF = pd.DataFrame(_stability_runs)
# BRANCH_STABILITY_SUMMARY_DF = (
#     BRANCH_STABILITY_DF.groupby('method', as_index=False)
#     .agg(
#         stability_support=('branch_detected', 'mean'),
#         median_x_coverage=('x_coverage', 'median'),
#         settings_tested=('setting', 'count'),
#     )
#     .sort_values(['stability_support', 'median_x_coverage'], ascending=False)
# )
#
# print('Target-panel parameter stability')
# display(BRANCH_STABILITY_DF.pivot(
#     index='setting', columns='method', values='branch_detected'
# ).fillna('').replace({True: 'Branch', False: 'No'}))
# display(BRANCH_STABILITY_SUMMARY_DF.round(3))

In [20]:
# LEGACY BRANCH BENCHMARK DISABLED: replaced by the Hexbin-only section below.
# Retained as comments for provenance; this cell intentionally performs no work.
# fig, axes = plt.subplots(2, 3, figsize=(15, 9), constrained_layout=True)
# axes = axes.ravel()
#
# x_limit = np.quantile(target_x, [0.005, 0.995])
# y_limit = np.quantile(target_y, [0.005, 0.995])
# for ax, result in zip(axes, TARGET_BRANCH_RESULTS):
#     plot_branch_result(ax, target_x, target_y, result, point_color='#4B74B5')
#     ax.set_xlim(x_limit)
#     ax.set_ylim(y_limit)
#     ax.set_xlabel('Precipitation, P (mm yr⁻¹)')
#     ax.set_ylabel('Total runoff, Q (mm yr⁻¹)')
#
# fig.suptitle(
#     'Static branch-method comparison · CMCC-CM2-SR5 · Land ice · P→Q',
#     fontsize=15, fontweight='bold',
# )
#
# BRANCH_FIGURE_PATH = FIGURE_DIR / 'S4_branch_method_comparison_CMCC_LI_P_Q.png'
# if WRITE_OUTPUTS:
#     fig.savefig(BRANCH_FIGURE_PATH, dpi=200, bbox_inches='tight')
#     print(f'Saved {BRANCH_FIGURE_PATH}')
# plt.show()
# plt.close(fig)
#
#
# target_detection = (
#     BRANCH_BENCHMARK_DF.loc[
#         BRANCH_BENCHMARK_DF['case'] == 'target_CMCC_LI_P_Q',
#         ['method', 'branch_detected', 'n_branches', 'x_coverage', 'notes'],
#     ]
#     .rename(columns={'branch_detected': 'target_detected'})
# )
#
# comparison_selectivity = (
#     BRANCH_BENCHMARK_DF.loc[
#         (BRANCH_BENCHMARK_DF['case_type'] == 'real')
#         & (BRANCH_BENCHMARK_DF['case'] != 'target_CMCC_LI_P_Q')
#     ]
#     .groupby('method', as_index=False)['branch_detected'].mean()
#     .rename(columns={'branch_detected': 'comparison_case_detection_rate'})
# )
#
# METHOD_COMPARISON_DF = (
#     BRANCH_SYNTHETIC_SCORES_DF
#     .merge(BRANCH_STABILITY_SUMMARY_DF, on='method', how='left')
#     .merge(target_detection, on='method', how='left')
#     .merge(comparison_selectivity, on='method', how='left')
# )
# METHOD_COMPARISON_DF['simplicity_1to5'] = METHOD_COMPARISON_DF['method'].map({
#     'Current conditional Dip': 5,
#     'Sliding KDE modal': 4,
#     'Sliding-window GMM': 5,
#     'Mixture of regressions': 4,
#     '2-D KDE density ridge': 2,
#     'Principal tree (MST)': 2,
# })
# METHOD_COMPARISON_DF['comparison_score'] = (
#     0.35 * METHOD_COMPARISON_DF['F1']
#     + 0.20 * METHOD_COMPARISON_DF['stability_support']
#     + 0.20 * METHOD_COMPARISON_DF['target_detected'].astype(float)
#     + 0.20 * (1 - METHOD_COMPARISON_DF['comparison_case_detection_rate'])
#     + 0.05 * METHOD_COMPARISON_DF['simplicity_1to5'] / 5
# )
# METHOD_COMPARISON_DF = METHOD_COMPARISON_DF.sort_values(
#     ['comparison_score', 'simplicity_1to5'], ascending=False
# )
#
# print('Combined comparison — use as a benchmark ranking, not a formal scientific test')
# display(METHOD_COMPARISON_DF[
#     ['method', 'target_detected', 'n_branches', 'F1', 'precision', 'recall',
#      'stability_support', 'median_x_coverage', 'comparison_case_detection_rate',
#      'simplicity_1to5', 'comparison_score']
# ].round(3))
#
# if WRITE_OUTPUTS:
#     benchmark_path = S4_OUTPUT_DIR / 'S4_branch_method_benchmark.csv'
#     stability_path = S4_OUTPUT_DIR / 'S4_branch_method_stability.csv'
#     comparison_path = S4_OUTPUT_DIR / 'S4_branch_method_summary.csv'
#     BRANCH_BENCHMARK_DF.to_csv(benchmark_path, index=False)
#     BRANCH_STABILITY_DF.to_csv(stability_path, index=False)
#     METHOD_COMPARISON_DF.to_csv(comparison_path, index=False)
#     print(f'Saved {benchmark_path}')
#     print(f'Saved {stability_path}')
#     print(f'Saved {comparison_path}')

## 11. Legacy six-method all-case batch — disabled

> This expensive and over-sensitive comparison is retained as commented code only. Its old outputs are cleared and it does not contribute to the active results.


In [21]:
# LEGACY BRANCH BENCHMARK DISABLED: replaced by the Hexbin-only section below.
# Retained as comments for provenance; this cell intentionally performs no work.
# import warnings as _branch_warnings
#
# ALL_BRANCH_MAX_POINTS = 6000
# PRIMARY_BRANCH_METHODS = [
#     'Mixture of regressions',
#     'Sliding KDE modal',
#     'Sliding-window GMM',
# ]
#
#
# def deterministic_x_subsample(x, y, max_points=ALL_BRANCH_MAX_POINTS):
#     x = np.asarray(x, dtype=float)
#     y = np.asarray(y, dtype=float)
#     finite = np.isfinite(x) & np.isfinite(y)
#     x, y = x[finite], y[finite]
#     if len(x) <= max_points:
#         return x, y
#     order = np.argsort(x)
#     take = np.linspace(0, len(order) - 1, max_points).round().astype(int)
#     selected = order[take]
#     return x[selected], y[selected]
#
#
# _all_branch_rows = []
# ALL_BRANCH_RESULT_OBJECTS = {}
# ALL_BRANCH_CASE_DATA = {}
# _case_counter = 0
# _total_cases = len(RUNS) * len(DOMAINS) * len(VARIABLE_PAIRS) * 2
# _batch_started = _branch_time.perf_counter()
#
# for run in RUNS:
#     frame = run['clim']
#     for domain_key, _, _ in DOMAINS:
#         if domain_key == 'all_land':
#             domain_frame = frame
#         else:
#             domain_frame = frame.loc[frame['analysis_zone'] == domain_key]
#         for x_var, y_var, pair_label in VARIABLE_PAIRS:
#             prepared = prepare_pair_versions(domain_frame[x_var], domain_frame[y_var])
#             for version in ['raw', 'trimmed']:
#                 x_values, y_values = deterministic_x_subsample(*prepared[version])
#                 key = (run['model'], domain_key, pair_label, version)
#                 started = _branch_time.perf_counter()
#                 with _branch_warnings.catch_warnings():
#                     _branch_warnings.simplefilter('ignore')
#                     results = run_all_branch_detectors(x_values, y_values)
#                 runtime = _branch_time.perf_counter() - started
#                 ALL_BRANCH_RESULT_OBJECTS[key] = results
#                 ALL_BRANCH_CASE_DATA[key] = (x_values, y_values)
#                 for result in results:
#                     row = result.to_dict()
#                     row.update({
#                         'model': run['model'],
#                         'domain': domain_key,
#                         'pair': pair_label,
#                         'version': version,
#                         'n_points_full': len(prepared[version][0]),
#                         'n_points_used': len(x_values),
#                         'case_runtime_s': runtime,
#                     })
#                     _all_branch_rows.append(row)
#                 _case_counter += 1
#                 if _case_counter % 20 == 0 or _case_counter == _total_cases:
#                     elapsed = _branch_time.perf_counter() - _batch_started
#                     print(f'  {_case_counter}/{_total_cases} cases · {elapsed:.1f}s elapsed')
#
# ALL_BRANCH_RESULTS_DF = pd.DataFrame(_all_branch_rows)
# print(
#     f'Completed {len(ALL_BRANCH_RESULTS_DF)} method-results '
#     f'for {_total_cases} cases in '
#     f'{_branch_time.perf_counter() - _batch_started:.1f}s'
# )

In [22]:
# LEGACY BRANCH BENCHMARK DISABLED: replaced by the Hexbin-only section below.
# Retained as comments for provenance; this cell intentionally performs no work.
# ALL_BRANCH_METHOD_SUMMARY_DF = (
#     ALL_BRANCH_RESULTS_DF
#     .groupby(['method', 'pair', 'version'], as_index=False)
#     .agg(
#         cases=('branch_detected', 'size'),
#         detected=('branch_detected', 'sum'),
#         detection_rate=('branch_detected', 'mean'),
#         median_n_branches=('n_branches', 'median'),
#         median_x_coverage=('x_coverage', 'median'),
#         median_runtime_s=('case_runtime_s', 'median'),
#     )
# )
#
# _raw_trim = ALL_BRANCH_RESULTS_DF.pivot_table(
#     index=['model', 'domain', 'pair', 'method'],
#     columns='version', values='branch_detected', aggfunc='first',
# ).reset_index()
# _raw_trim['consistent'] = _raw_trim['raw'] == _raw_trim['trimmed']
# _raw_trim['raw_to_trim_flip'] = _raw_trim['raw'].astype(int) - _raw_trim['trimmed'].astype(int)
#
# ALL_BRANCH_RAW_TRIM_CONSISTENCY_DF = (
#     _raw_trim.groupby(['method', 'pair'], as_index=False)
#     .agg(
#         cases=('consistent', 'size'),
#         consistency_rate=('consistent', 'mean'),
#         raw_detection_rate=('raw', 'mean'),
#         trimmed_detection_rate=('trimmed', 'mean'),
#         changed_cases=('consistent', lambda values: int((~values).sum())),
#     )
# )
#
#
# def summarize_case_votes(group):
#     detected_methods = group.loc[group['branch_detected'], 'method'].tolist()
#     primary_detected = [m for m in detected_methods if m in PRIMARY_BRANCH_METHODS]
#     return pd.Series({
#         'all_method_votes': len(detected_methods),
#         'primary_votes': len(primary_detected),
#         'detected_methods': ' | '.join(detected_methods),
#         'primary_detected_methods': ' | '.join(primary_detected),
#         'primary_disagreement': 0 < len(primary_detected) < len(PRIMARY_BRANCH_METHODS),
#     })
#
#
# ALL_BRANCH_CASE_VOTES_DF = (
#     ALL_BRANCH_RESULTS_DF
#     .groupby(['model', 'domain', 'pair', 'version'])
#     .apply(summarize_case_votes, include_groups=False)
#     .reset_index()
# )
# ALL_BRANCH_DISAGREEMENTS_DF = ALL_BRANCH_CASE_VOTES_DF.loc[
#     ALL_BRANCH_CASE_VOTES_DF['primary_disagreement']
# ].copy()
#
# print('Detection rate by method, variable pair, and version')
# display(ALL_BRANCH_METHOD_SUMMARY_DF.round(3))
# print('Raw/trimmed consistency')
# display(ALL_BRANCH_RAW_TRIM_CONSISTENCY_DF.round(3))
# print(f'Primary-method disagreement cases: {len(ALL_BRANCH_DISAGREEMENTS_DF)}')
# display(ALL_BRANCH_DISAGREEMENTS_DF.head(30))
#
# if WRITE_OUTPUTS:
#     ALL_BRANCH_RESULTS_DF.to_csv(
#         S4_OUTPUT_DIR / 'S4_branch_all_cases_results.csv', index=False,
#     )
#     ALL_BRANCH_METHOD_SUMMARY_DF.to_csv(
#         S4_OUTPUT_DIR / 'S4_branch_all_cases_method_summary.csv', index=False,
#     )
#     ALL_BRANCH_RAW_TRIM_CONSISTENCY_DF.to_csv(
#         S4_OUTPUT_DIR / 'S4_branch_all_cases_raw_trim_consistency.csv', index=False,
#     )
#     ALL_BRANCH_CASE_VOTES_DF.to_csv(
#         S4_OUTPUT_DIR / 'S4_branch_all_cases_votes.csv', index=False,
#     )
#     ALL_BRANCH_DISAGREEMENTS_DF.to_csv(
#         S4_OUTPUT_DIR / 'S4_branch_all_cases_disagreements.csv', index=False,
#     )

In [23]:
# LEGACY BRANCH BENCHMARK DISABLED: replaced by the Hexbin-only section below.
# Retained as comments for provenance; this cell intentionally performs no work.
# from matplotlib.colors import ListedColormap
#
# _branch_binary_cmap = ListedColormap(['#F3F4F6', '#2563EB'])
# _model_order = [run['model'] for run in RUNS]
# _domain_order = [item[0] for item in DOMAINS]
# _method_order = [
#     'Current conditional Dip',
#     'Sliding KDE modal',
#     'Sliding-window GMM',
#     'Mixture of regressions',
#     '2-D KDE density ridge',
#     'Principal tree (MST)',
# ]
#
#
# def draw_all_case_branch_heatmap(pair_label, version):
#     subset = ALL_BRANCH_RESULTS_DF.loc[
#         (ALL_BRANCH_RESULTS_DF['pair'] == pair_label)
#         & (ALL_BRANCH_RESULTS_DF['version'] == version)
#     ]
#     fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
#     for ax, method in zip(axes.ravel(), _method_order):
#         matrix = (
#             subset.loc[subset['method'] == method]
#             .pivot(index='model', columns='domain', values='branch_detected')
#             .reindex(index=_model_order, columns=_domain_order)
#             .fillna(False).astype(int)
#         )
#         ax.imshow(matrix, vmin=0, vmax=1, cmap=_branch_binary_cmap, aspect='auto')
#         for row_i in range(matrix.shape[0]):
#             for col_i in range(matrix.shape[1]):
#                 value = int(matrix.iloc[row_i, col_i])
#                 ax.text(
#                     col_i, row_i, 'B' if value else '–',
#                     ha='center', va='center', fontsize=10,
#                     color='white' if value else '#4B5563', fontweight='bold',
#                 )
#         ax.set_xticks(range(len(_domain_order)), _domain_order, rotation=35, ha='right')
#         ax.set_yticks(range(len(_model_order)), _model_order)
#         rate = matrix.to_numpy().mean()
#         ax.set_title(f'{method}\nBranch rate={rate:.2f}', fontsize=10, fontweight='bold')
#         ax.grid(False)
#     fig.suptitle(
#         f'Branch detection heatmaps · {pair_label} · {version.upper()}',
#         fontsize=15, fontweight='bold',
#     )
#     output_path = FIGURE_DIR / (
#         f'S4_branch_heatmap_{PAIR_SLUGS[pair_label]}_{version}.png'
#     )
#     if WRITE_OUTPUTS:
#         fig.savefig(output_path, dpi=180, bbox_inches='tight')
#         print(f'Saved {output_path}')
#     plt.show()
#     plt.close(fig)
#
#
# for _, _, pair_label in VARIABLE_PAIRS:
#     for version in ['raw', 'trimmed']:
#         draw_all_case_branch_heatmap(pair_label, version)

In [24]:
# LEGACY BRANCH BENCHMARK DISABLED: replaced by the Hexbin-only section below.
# Retained as comments for provenance; this cell intentionally performs no work.
# BRANCH_CASE_FIGURE_DIR = FIGURE_DIR / 'branch_cases'
# if WRITE_OUTPUTS:
#     BRANCH_CASE_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
#
# _selected_cases = ALL_BRANCH_CASE_VOTES_DF.loc[
#     ALL_BRANCH_CASE_VOTES_DF['primary_votes'] > 0
# ].copy()
# _saved_case_figures = []
#
# for figure_i, row in enumerate(_selected_cases.itertuples(index=False), start=1):
#     key = (row.model, row.domain, row.pair, row.version)
#     x_values, y_values = ALL_BRANCH_CASE_DATA[key]
#     results = ALL_BRANCH_RESULT_OBJECTS[key]
#     fig, axes = plt.subplots(2, 3, figsize=(14, 8.5), constrained_layout=True)
#     x_limit = np.quantile(x_values, [0.005, 0.995])
#     y_limit = np.quantile(y_values, [0.005, 0.995])
#     for ax, result in zip(axes.ravel(), results):
#         plot_branch_result(ax, x_values, y_values, result, point_color='#64748B')
#         ax.set_xlim(x_limit)
#         ax.set_ylim(y_limit)
#         ax.set_xlabel(row.pair.split(' → ')[0])
#         ax.set_ylabel(row.pair.split(' → ')[1])
#     fig.suptitle(
#         f'{row.model} · {row.domain} · {row.pair} · {row.version.upper()} '
#         f'· primary votes={row.primary_votes}/3',
#         fontsize=14, fontweight='bold',
#     )
#     filename = (
#         f"{row.model}_{row.domain}_{PAIR_SLUGS[row.pair]}_{row.version}.png"
#         .replace('/', '_').replace(' ', '_')
#     )
#     output_path = BRANCH_CASE_FIGURE_DIR / filename
#     if WRITE_OUTPUTS:
#         fig.savefig(output_path, dpi=150, bbox_inches='tight')
#     plt.close(fig)
#     _saved_case_figures.append({
#         'model': row.model, 'domain': row.domain, 'pair': row.pair,
#         'version': row.version, 'primary_votes': row.primary_votes,
#         'all_method_votes': row.all_method_votes, 'figure_path': str(output_path),
#     })
#     if figure_i % 20 == 0 or figure_i == len(_selected_cases):
#         print(f'  saved {figure_i}/{len(_selected_cases)} branch comparison figures')
#
# ALL_BRANCH_CASE_FIGURES_DF = pd.DataFrame(_saved_case_figures)
# if WRITE_OUTPUTS:
#     ALL_BRANCH_CASE_FIGURES_DF.to_csv(
#         S4_OUTPUT_DIR / 'S4_branch_all_cases_figure_manifest.csv', index=False,
#     )
#
# print(f'Cases detected by at least one primary method: {len(_selected_cases)}')
# print(f'Cases with all three primary methods agreeing on Branch: '
#       f'{int((_selected_cases.primary_votes == 3).sum())}')
# display(ALL_BRANCH_CASE_FIGURES_DF.head(30))

## 12. Legacy branch detector: multi-extent Hexbin — disabled

This earlier Hexbin implementation is retained for reference but its code cells are disabled. The active branch calculations now begin in Sections 14–15.

The retained code is not executed and does not contribute to the current results.


In [25]:
# Legacy multi-extent Hexbin code retained below; execution disabled.
# import importlib
# import sys

# if str(CASE_DIR) not in sys.path:
#     sys.path.insert(0, str(CASE_DIR))

# import branch_benchmark as _branch_benchmark
# _branch_benchmark = importlib.reload(_branch_benchmark)
# from branch_benchmark import detect_multiextent_hexbin, plot_branch_result

# HEXBIN_GRIDSIZES = (16, 22, 28, 34, 40, 46)
# HEXBIN_EXTENT_QUANTILES = (0.0, 0.0025)
# HEXBIN_REQUIRED_SUPPORT = 3

# HEXBIN_CASE_DATA = {}
# HEXBIN_RESULT_OBJECTS = {}
# _hexbin_rows = []
# _hexbin_started = _time.perf_counter()

# for run in RUNS:
#     model = run['model']
#     frame = run['clim']
#     for x_var, y_var, pair_label in VARIABLE_PAIRS:
#         x_all = np.asarray(frame[x_var], dtype=float)
#         y_all = np.asarray(frame[y_var], dtype=float)
#         finite_all = np.isfinite(x_all) & np.isfinite(y_all)
#         x_lo, x_hi = np.quantile(x_all[finite_all], [0.01, 0.99])
#         y_lo, y_hi = np.quantile(y_all[finite_all], [0.01, 0.99])

#         for domain_key, _, domain_filter in DOMAINS:
#             subset = frame if domain_filter is None else frame.loc[domain_filter(frame)]
#             x_domain = np.asarray(subset[x_var], dtype=float)
#             y_domain = np.asarray(subset[y_var], dtype=float)
#             finite = np.isfinite(x_domain) & np.isfinite(y_domain)
#             trim_keep = (
#                 finite
#                 & (x_domain >= x_lo) & (x_domain <= x_hi)
#                 & (y_domain >= y_lo) & (y_domain <= y_hi)
#             )

#             for version, keep in [('raw', finite), ('trimmed', trim_keep)]:
#                 x_values = x_domain[keep]
#                 y_values = y_domain[keep]
#                 result = detect_multiextent_hexbin(
#                     x_values,
#                     y_values,
#                     extent_quantiles=HEXBIN_EXTENT_QUANTILES,
#                     gridsizes=HEXBIN_GRIDSIZES,
#                     required_support=HEXBIN_REQUIRED_SUPPORT,
#                 )
#                 grid_support = int(round(result.score * len(HEXBIN_GRIDSIZES)))
#                 if result.branch_detected:
#                     branch_status, status_code = 'Branch', 2
#                 elif grid_support >= 1:
#                     branch_status, status_code = 'Candidate', 1
#                 else:
#                     branch_status, status_code = 'No branch', 0
#                 extent_runs = (result.geometry or {}).get('extent_runs', [])
#                 extent_votes = int(sum(item['branch_detected'] for item in extent_runs))
#                 selected_clip = (result.geometry or {}).get('selected_extent_quantile', np.nan)
#                 key = (model, domain_key, pair_label, version)
#                 HEXBIN_CASE_DATA[key] = (x_values, y_values)
#                 HEXBIN_RESULT_OBJECTS[key] = result
#                 _hexbin_rows.append({
#                     'model': model,
#                     'domain': domain_key,
#                     'pair': pair_label,
#                     'version': version,
#                     'n_points': len(x_values),
#                     'branch_status': branch_status,
#                     'status_code': status_code,
#                     'grid_support': grid_support,
#                     'extent_votes': extent_votes,
#                     'selected_clip_quantile': selected_clip,
#                     **result.to_dict(),
#                 })

# HEXBIN_RESULTS_DF = pd.DataFrame(_hexbin_rows)
# print(
#     f'Multi-extent Hexbin completed: {len(HEXBIN_RESULTS_DF)} panels in '
#     f'{_time.perf_counter() - _hexbin_started:.1f}s'
# )


In [26]:
# Legacy multi-extent Hexbin summary retained below; execution disabled.
# HEXBIN_SUMMARY_DF = (
#     HEXBIN_RESULTS_DF
#     .groupby(['pair', 'version'], as_index=False)
#     .agg(
#         panels=('branch_status', 'size'),
#         branches=('branch_status', lambda values: int((values == 'Branch').sum())),
#         candidates=('branch_status', lambda values: int((values == 'Candidate').sum())),
#         median_grid_support=('grid_support', 'median'),
#         median_x_coverage=('x_coverage', 'median'),
#     )
# )

# HEXBIN_DETECTIONS_DF = (
#     HEXBIN_RESULTS_DF.loc[
#         HEXBIN_RESULTS_DF['branch_status'] != 'No branch',
#         ['model', 'domain', 'pair', 'version', 'branch_status', 'n_points',
#          'grid_support', 'extent_votes', 'selected_clip_quantile',
#          'x_coverage', 'min_branch_weight', 'branch_separation'],
#     ]
#     .sort_values(['branch_status', 'pair', 'version', 'model', 'domain'])
# )
# HEXBIN_BRANCHES_DF = HEXBIN_DETECTIONS_DF.loc[
#     HEXBIN_DETECTIONS_DF['branch_status'] == 'Branch'
# ].copy()
# HEXBIN_CANDIDATES_DF = HEXBIN_DETECTIONS_DF.loc[
#     HEXBIN_DETECTIONS_DF['branch_status'] == 'Candidate'
# ].copy()

# print('Multi-extent Hexbin summary')
# display(HEXBIN_SUMMARY_DF.round(3))
# print(f'Branch panels: {len(HEXBIN_BRANCHES_DF)}')
# display(HEXBIN_BRANCHES_DF.round(3))
# print(f'Candidate panels: {len(HEXBIN_CANDIDATES_DF)}')
# display(HEXBIN_CANDIDATES_DF.round(3))

# if WRITE_OUTPUTS:
#     HEXBIN_RESULTS_DF.to_csv(
#         S4_OUTPUT_DIR / 'S4_branch_hexbin_all_cases_results.csv', index=False,
#     )
#     HEXBIN_SUMMARY_DF.to_csv(
#         S4_OUTPUT_DIR / 'S4_branch_hexbin_all_cases_summary.csv', index=False,
#     )
#     HEXBIN_BRANCHES_DF.to_csv(
#         S4_OUTPUT_DIR / 'S4_branch_hexbin_detected_cases.csv', index=False,
#     )
#     HEXBIN_CANDIDATES_DF.to_csv(
#         S4_OUTPUT_DIR / 'S4_branch_hexbin_candidate_cases.csv', index=False,
#     )
#     HEXBIN_RESULTS_DF.loc[HEXBIN_RESULTS_DF['pair'] == 'P → Q'].to_csv(
#         S4_OUTPUT_DIR / 'S4_branch_hexbin_P_Q_results.csv', index=False,
#     )


In [27]:
# Legacy multi-extent Hexbin figures retained below; execution disabled.
# from matplotlib.colors import ListedColormap

# Compact all-case Branch/Candidate heatmap.
# _hexbin_status_cmap = ListedColormap(['#F3F4F6', '#F59E0B', '#2563EB'])
# fig, axes = plt.subplots(2, 3, figsize=(14, 7.5), constrained_layout=True)
# for row_index, version in enumerate(['raw', 'trimmed']):
#     for col_index, (_, _, pair_label) in enumerate(VARIABLE_PAIRS):
#         ax = axes[row_index, col_index]
#         matrix = (
#             HEXBIN_RESULTS_DF.loc[
#                 (HEXBIN_RESULTS_DF['pair'] == pair_label)
#                 & (HEXBIN_RESULTS_DF['version'] == version)
#             ]
#             .pivot(index='model', columns='domain', values='status_code')
#             .reindex(index=MODELS, columns=[item[0] for item in DOMAINS])
#             .fillna(0).astype(int)
#         )
#         ax.imshow(matrix, vmin=0, vmax=2, cmap=_hexbin_status_cmap, aspect='auto')
#         for row_i in range(matrix.shape[0]):
#             for col_i in range(matrix.shape[1]):
#                 value = int(matrix.iloc[row_i, col_i])
#                 ax.text(
#                     col_i, row_i, {0: '–', 1: 'C', 2: 'B'}[value],
#                     ha='center', va='center', fontsize=10,
#                     color='white' if value else '#4B5563', fontweight='bold',
#                 )
#         ax.set_xticks(range(len(DOMAINS)), [item[0] for item in DOMAINS], rotation=35, ha='right')
#         ax.set_yticks(range(len(MODELS)), MODELS)
#         n_branch = int((matrix.to_numpy() == 2).sum())
#         n_candidate = int((matrix.to_numpy() == 1).sum())
#         ax.set_title(
#             f'{pair_label} · {version.upper()} · B={n_branch}, C={n_candidate}',
#             fontsize=10, fontweight='bold',
#         )
#         ax.grid(False)
# fig.suptitle('Multi-extent Hexbin: Branch (B) and Candidate (C)', fontsize=15, fontweight='bold')
# HEXBIN_HEATMAP_PATH = FIGURE_DIR / 'S4_branch_hexbin_all_cases_heatmap.png'
# if WRITE_OUTPUTS:
#     fig.savefig(HEXBIN_HEATMAP_PATH, dpi=180, bbox_inches='tight')
#     print(f'Saved {HEXBIN_HEATMAP_PATH}')
# plt.show()
# plt.close(fig)


# def draw_hexbin_case_grid(pair_label, version, show=False):
#     fig, axes = plt.subplots(5, 6, figsize=(18, 14), constrained_layout=True)
#     for row_index, model in enumerate(MODELS):
#         for col_index, (domain_key, domain_label, _) in enumerate(DOMAINS):
#             ax = axes[row_index, col_index]
#             key = (model, domain_key, pair_label, version)
#             x_values, y_values = HEXBIN_CASE_DATA[key]
#             result = HEXBIN_RESULT_OBJECTS[key]
#             row = HEXBIN_RESULTS_DF.loc[
#                 (HEXBIN_RESULTS_DF['model'] == model)
#                 & (HEXBIN_RESULTS_DF['domain'] == domain_key)
#                 & (HEXBIN_RESULTS_DF['pair'] == pair_label)
#                 & (HEXBIN_RESULTS_DF['version'] == version)
#             ].iloc[0]
#             plot_branch_result(ax, x_values, y_values, result)
#             status = row['branch_status']
#             color = {'Branch': '#B91C1C', 'Candidate': '#B45309', 'No branch': '#374151'}[status]
#             ax.set_title(
#                 f"{model} · {domain_label}\n{status} · support {int(row['grid_support'])}/{len(HEXBIN_GRIDSIZES)}",
#                 fontsize=9, color=color, fontweight='bold',
#             )
#             if row_index == len(MODELS) - 1:
#                 ax.set_xlabel(pair_label.split(' → ')[0], fontsize=8)
#             if col_index == 0:
#                 ax.set_ylabel(pair_label.split(' → ')[1], fontsize=8)
#             ax.tick_params(labelsize=7)
#     fig.suptitle(
#         f'{pair_label} · {version.upper()} · multi-extent Hexbin branch detector',
#         fontsize=16, fontweight='bold',
#     )
#     output_path = FIGURE_DIR / f'S4_branch_hexbin_{PAIR_SLUGS[pair_label]}_{version}.png'
#     if WRITE_OUTPUTS:
#         fig.savefig(output_path, dpi=180, bbox_inches='tight')
#     if show:
#         print(f'Saved {output_path}')
#         plt.show()
#     plt.close(fig)
#     return output_path


# HEXBIN_GRID_PATHS = {}
# for _, _, pair_label in VARIABLE_PAIRS:
#     for version in ['raw', 'trimmed']:
#         key = (pair_label, version)
#         HEXBIN_GRID_PATHS[key] = draw_hexbin_case_grid(
#             pair_label, version,
#             show=(pair_label == 'P → Q'),
#         )


# Local threshold audit without visual labels.
# _hexbin_stability_rows = []
# for valley_depth in [0.30, 0.35, 0.40]:
#     for relative_peak in [0.13, 0.15, 0.17]:
#         for model in MODELS:
#             for domain_key, _, _ in DOMAINS:
#                 for version in ['raw', 'trimmed']:
#                     key = (model, domain_key, 'P → Q', version)
#                     x_values, y_values = HEXBIN_CASE_DATA[key]
#                     result = detect_multiextent_hexbin(
#                         x_values, y_values,
#                         extent_quantiles=HEXBIN_EXTENT_QUANTILES,
#                         gridsizes=HEXBIN_GRIDSIZES,
#                         required_support=HEXBIN_REQUIRED_SUPPORT,
#                         min_valley_depth=valley_depth,
#                         min_relative_peak=relative_peak,
#                     )
#                     _hexbin_stability_rows.append({
#                         'model': model,
#                         'domain': domain_key,
#                         'version': version,
#                         'min_valley_depth': valley_depth,
#                         'min_relative_peak': relative_peak,
#                         'branch_detected': result.branch_detected,
#                         'grid_support': int(round(result.score * len(HEXBIN_GRIDSIZES))),
#                     })

# HEXBIN_PQ_STABILITY_DF = pd.DataFrame(_hexbin_stability_rows)
# HEXBIN_PQ_STABILITY_SUMMARY_DF = (
#     HEXBIN_PQ_STABILITY_DF.groupby(['model', 'domain', 'version'], as_index=False)
#     .agg(
#         detected_settings=('branch_detected', 'sum'),
#         total_settings=('branch_detected', 'size'),
#     )
# )
# HEXBIN_PQ_STABILITY_SUMMARY_DF['detection_rate'] = (
#     HEXBIN_PQ_STABILITY_SUMMARY_DF['detected_settings']
#     / HEXBIN_PQ_STABILITY_SUMMARY_DF['total_settings']
# )
# print('P→Q threshold sensitivity: panels detected in at least one setting')
# display(HEXBIN_PQ_STABILITY_SUMMARY_DF.loc[
#     HEXBIN_PQ_STABILITY_SUMMARY_DF['detected_settings'] > 0
# ].round(3))

# if WRITE_OUTPUTS:
#     HEXBIN_PQ_STABILITY_DF.to_csv(
#         S4_OUTPUT_DIR / 'S4_branch_hexbin_P_Q_threshold_stability.csv', index=False,
#     )
#     HEXBIN_PQ_STABILITY_SUMMARY_DF.to_csv(
#         S4_OUTPUT_DIR / 'S4_branch_hexbin_P_Q_threshold_summary.csv', index=False,
#     )


## 13. Legacy Hexbin cross-window tracking — disabled

This earlier tracked-Hexbin refinement is retained for reference but is not executed.


In [28]:
# Legacy tracked-Hexbin code retained below; execution disabled.
# from branch_benchmark import detect_tracked_multiextent_hexbin

# TRACKED_RESULT_OBJECTS = {}
# _tracked_rows = []
# _tracked_started = _time.perf_counter()
# for key, (x_values, y_values) in HEXBIN_CASE_DATA.items():
#     model, domain_key, pair_label, version = key
#     result = detect_tracked_multiextent_hexbin(
#         x_values, y_values,
#         extent_quantiles=HEXBIN_EXTENT_QUANTILES,
#         gridsizes=HEXBIN_GRIDSIZES,
#         required_support=HEXBIN_REQUIRED_SUPPORT,
#     )
#     geometry = result.geometry or {}
#     best_track = geometry.get('best_track') or {}
#     status = geometry.get('track_status', 'No branch')
#     status_code = {'No branch': 0, 'Candidate': 1, 'Two-band': 2, 'Branch': 3}[status]
#     TRACKED_RESULT_OBJECTS[key] = result
#     _tracked_rows.append({
#         'model': model, 'domain': domain_key, 'pair': pair_label, 'version': version,
#         'n_points': len(x_values), 'track_status': status, 'status_code': status_code,
#         'branch_support': int(geometry.get('branch_support', 0)),
#         'band_support': int(geometry.get('band_support', 0)),
#         'selected_clip_quantile': geometry.get('selected_extent_quantile', np.nan),
#         'track_windows': int(best_track.get('n_windows', 0)),
#         'jump_q90': best_track.get('jump_q90', np.nan),
#         'curve_rmse': best_track.get('curve_rmse', np.nan),
#         'separation_change': best_track.get('separation_change', np.nan),
#         'relative_separation_change': best_track.get('relative_separation_change', np.nan),
#         'separation_correlation': best_track.get('separation_correlation', np.nan),
#         **result.to_dict(),
#     })

# TRACKED_RESULTS_DF = pd.DataFrame(_tracked_rows)
# TRACKED_SUMMARY_DF = (
#     TRACKED_RESULTS_DF.groupby(['pair', 'version', 'track_status']).size()
#     .rename('n_panels').reset_index()
# )
# print(f'Tracked Hexbin completed: {len(TRACKED_RESULTS_DF)} panels in {_time.perf_counter() - _tracked_started:.1f}s')
# display(TRACKED_SUMMARY_DF.pivot_table(index=['pair', 'version'], columns='track_status', values='n_panels', fill_value=0).astype(int))

# _baseline = HEXBIN_RESULTS_DF[['model', 'domain', 'pair', 'version', 'branch_status']].rename(columns={'branch_status': 'local_peak_status'})
# TRACKED_COMPARISON_DF = _baseline.merge(
#     TRACKED_RESULTS_DF[['model', 'domain', 'pair', 'version', 'track_status']],
#     on=['model', 'domain', 'pair', 'version'], how='inner', validate='one_to_one',
# )
# print('Original local-peak status → tracked status')
# display(pd.crosstab(TRACKED_COMPARISON_DF['local_peak_status'], TRACKED_COMPARISON_DF['track_status']))

# if WRITE_OUTPUTS:
#     TRACKED_RESULTS_DF.to_csv(S4_OUTPUT_DIR / 'S4_branch_tracked_all_cases_results.csv', index=False)
#     TRACKED_SUMMARY_DF.to_csv(S4_OUTPUT_DIR / 'S4_branch_tracked_all_cases_summary.csv', index=False)
#     TRACKED_COMPARISON_DF.to_csv(S4_OUTPUT_DIR / 'S4_branch_tracked_vs_local_comparison.csv', index=False)

# _tracked_cmap = ListedColormap(['#F3F4F6', '#F59E0B', '#7C3AED', '#2563EB'])
# _tracked_symbols = {0: '–', 1: 'C', 2: 'T', 3: 'B'}
# fig, axes = plt.subplots(2, 3, figsize=(14, 7.5), constrained_layout=True)
# for row_index, version in enumerate(['raw', 'trimmed']):
#     for col_index, (_, _, pair_label) in enumerate(VARIABLE_PAIRS):
#         ax = axes[row_index, col_index]
#         matrix = (
#             TRACKED_RESULTS_DF.loc[(TRACKED_RESULTS_DF['pair'] == pair_label) & (TRACKED_RESULTS_DF['version'] == version)]
#             .pivot(index='model', columns='domain', values='status_code')
#             .reindex(index=MODELS, columns=[item[0] for item in DOMAINS]).fillna(0).astype(int)
#         )
#         ax.imshow(matrix, vmin=0, vmax=3, cmap=_tracked_cmap, aspect='auto')
#         for row_i in range(matrix.shape[0]):
#             for col_i in range(matrix.shape[1]):
#                 value = int(matrix.iloc[row_i, col_i])
#                 ax.text(col_i, row_i, _tracked_symbols[value], ha='center', va='center', fontsize=10,
#                         color='white' if value else '#4B5563', fontweight='bold')
#         ax.set_xticks(range(len(DOMAINS)), [item[0] for item in DOMAINS], rotation=35, ha='right')
#         ax.set_yticks(range(len(MODELS)), MODELS)
#         counts = matrix.stack().value_counts()
#         ax.set_title(f"{pair_label} · {version.upper()} · B={counts.get(3, 0)}, T={counts.get(2, 0)}, C={counts.get(1, 0)}", fontsize=10, fontweight='bold')
#         ax.grid(False)
# fig.suptitle('Tracked Hexbin: Branch (B), Two-band (T), Candidate (C)', fontsize=15, fontweight='bold')
# TRACKED_HEATMAP_PATH = FIGURE_DIR / 'S4_branch_tracked_all_cases_heatmap.png'
# if WRITE_OUTPUTS:
#     fig.savefig(TRACKED_HEATMAP_PATH, dpi=180, bbox_inches='tight')
# plt.show()
# plt.close(fig)

# def draw_tracked_case_grid(pair_label, version, show=False):
#     fig, axes = plt.subplots(5, 6, figsize=(18, 14), constrained_layout=True)
#     status_colors = {'Branch': '#B91C1C', 'Two-band': '#6D28D9', 'Candidate': '#B45309', 'No branch': '#374151'}
#     for row_index, model in enumerate(MODELS):
#         for col_index, (domain_key, domain_label, _) in enumerate(DOMAINS):
#             ax = axes[row_index, col_index]
#             key = (model, domain_key, pair_label, version)
#             x_values, y_values = HEXBIN_CASE_DATA[key]
#             result = TRACKED_RESULT_OBJECTS[key]
#             row = TRACKED_RESULTS_DF.loc[
#                 (TRACKED_RESULTS_DF['model'] == model) & (TRACKED_RESULTS_DF['domain'] == domain_key)
#                 & (TRACKED_RESULTS_DF['pair'] == pair_label) & (TRACKED_RESULTS_DF['version'] == version)
#             ].iloc[0]
#             plot_branch_result(ax, x_values, y_values, result)
#             status = row['track_status']
#             ax.set_title(
#                 f"{model} · {domain_label}\n{status} · branch {int(row['branch_support'])}/{len(HEXBIN_GRIDSIZES)} · tracks {int(row['band_support'])}/{len(HEXBIN_GRIDSIZES)}",
#                 fontsize=8.5, color=status_colors[status], fontweight='bold',
#             )
#             if row_index == len(MODELS) - 1:
#                 ax.set_xlabel(pair_label.split(' → ')[0], fontsize=8)
#             if col_index == 0:
#                 ax.set_ylabel(pair_label.split(' → ')[1], fontsize=8)
#             ax.tick_params(labelsize=7)
#     fig.suptitle(f'{pair_label} · {version.upper()} · cross-window tracked Hexbin', fontsize=16, fontweight='bold')
#     output_path = FIGURE_DIR / f'S4_branch_tracked_{PAIR_SLUGS[pair_label]}_{version}.png'
#     if WRITE_OUTPUTS:
#         fig.savefig(output_path, dpi=180, bbox_inches='tight')
#     if show:
#         plt.show()
#     plt.close(fig)
#     return output_path

# TRACKED_GRID_PATHS = {}
# for _, _, pair_label in VARIABLE_PAIRS:
#     for version in ['raw', 'trimmed']:
#         TRACKED_GRID_PATHS[(pair_label, version)] = draw_tracked_case_grid(
#             pair_label, version, show=(pair_label == 'P → Q'),
#         )


## 14. Branch detection A — scatter-window conditional KDE / 散点窗口 KDE

Association gate first, then branch classification. Both MIC and dCor are computed; `BRANCH_GATE_MODE` chooses which gate opens the detector (`mic`, `dcor`, `both`, `either`, or `none`). Panels below the gate are labelled **No global** and are not branch-classified.

On panels that pass: equal-count x windows, five-bandwidth 1-D KDE of y, and lower-peak–valley–upper-peak tracks across neighboring windows. Local geometry: relative valley depth ≥ 0.30, smaller-side mass ≥ 0.10, peak separation ≥ 0.80 robust y scales. Candidate needs at least two connected windows; three or more are Branch when the separation opens/closes, otherwise Two-band.

**中文说明：** 先过 MIC / dCor gate，通过的 panel 才做 branch 分类。两个分数都保留、都画在图上；用 `BRANCH_GATE_MODE` 切换 gate。没过门槛的标成 No global，不再检测分叉。


In [ ]:
import importlib
import sys

if str(CASE_DIR) not in sys.path:
    sys.path.insert(0, str(CASE_DIR))

from test_residual_dip import build_cases as _build_branch_cases
from test_residual_dip import load_runs as _load_branch_runs
from IPython.display import Image as _NotebookImage
import test_two_condition_kde as _scatter_branch_detector
import kde_branch_comparison as _kde_branch_comparison

_scatter_branch_detector = importlib.reload(_scatter_branch_detector)
_kde_branch_comparison = importlib.reload(_kde_branch_comparison)
BRANCH_KDE_CASES = _build_branch_cases(_load_branch_runs(), pairs=VARIABLE_PAIRS)

# Association gate that must pass before the KDE branch detector runs.
# Both MIC and dCor are always computed and drawn on the panels.
#   'mic'    — MIC ≥ BRANCH_MIC_MIN
#   'dcor'   — dCor ≥ BRANCH_DCOR_MIN
#   'both'   — MIC and dCor both pass (AND)
#   'either' — MIC or dCor passes (OR)
#   'none'   — no association gate; classify every panel
BRANCH_GATE_MODE = 'mic'
# BRANCH_MIC_MIN = CLASSIFIER_DEFAULTS['mic_intermediate']
# BRANCH_DCOR_MIN = CLASSIFIER_DEFAULTS['dcor_intermediate']
BRANCH_MIC_MIN = 0.2
BRANCH_DCOR_MIN = 0.4
BRANCH_COVERAGE_MODE = CLASSIFIER_DEFAULTS['branch_coverage_mode']
DETECT_BEHIND_GATE = CLASSIFIER_DEFAULTS['detect_behind_gate']

_association_required = {'model', 'domain', 'pair', 'outlier_version', 'mic', 'dcor'}
_association_source = globals().get('RESULTS_DF')
_association_source_name = 'current RESULTS_DF'
if (
    not isinstance(_association_source, pd.DataFrame)
    or not _association_required.issubset(_association_source.columns)
):
    _association_path = (
        S4_OUTPUT_DIR
        / f'S4_classification_results_{START_YEAR}_{END_YEAR}.parquet'
    )
    if not _association_path.exists():
        missing = sorted(
            _association_required
            - set(getattr(_association_source, 'columns', []))
        )
        raise FileNotFoundError(
            f'Cannot build MIC/dCor filter: RESULTS_DF is missing {missing} '
            f'and saved metrics do not exist at {_association_path}'
        )
    _association_source = pd.read_parquet(_association_path)
    _association_source_name = str(_association_path)

_BRANCH_ASSOCIATION_METRICS = (
    _association_source.loc[
        _association_source['sample'].eq('all'),
        ['model', 'domain', 'pair', 'outlier_version', 'mic', 'dcor'],
    ]
    .rename(columns={'outlier_version': 'version'})
    .drop_duplicates(['model', 'domain', 'pair', 'version'])
)
print(
    f'Association metrics: {_association_source_name}\n'
    f'Branch gate: {BRANCH_GATE_MODE} · '
    f'MIC ≥ {BRANCH_MIC_MIN:.2f} · dCor ≥ {BRANCH_DCOR_MIN:.2f}\n'
    f'Panels below the gate are labelled No global and are not branch-classified.'
)

(
    SCATTER_KDE_RESULTS_DF,
    SCATTER_KDE_RESULT_OBJECTS,
    SCATTER_KDE_GEOMETRY,
    _scatter_kde_elapsed,
) = _kde_branch_comparison.run_detector(
    BRANCH_KDE_CASES,
    method='scatter',
    metrics=_BRANCH_ASSOCIATION_METRICS,
    mode=BRANCH_GATE_MODE,
    mic_min=BRANCH_MIC_MIN,
    dcor_min=BRANCH_DCOR_MIN,
    coverage_mode=BRANCH_COVERAGE_MODE,
    detect_behind_gate=DETECT_BEHIND_GATE,
)

print(
    f'Scatter-window KDE completed: {len(SCATTER_KDE_RESULTS_DF)} panels '
    f'in {_scatter_kde_elapsed:.1f}s'
)
display(_kde_branch_comparison.summarize(SCATTER_KDE_RESULTS_DF))



## 15. Branch detection B — conditional 2-D KDE / 二维 KDE 条件密度

Same MIC / dCor gate as Section 14. Panels that pass are smoothed into a 2-D KDE surface; each vertical column is normalized to p(y | x), and the same peak–valley geometry is connected across adjacent columns.

**中文说明：** 和第 14 节共用同一套 MIC / dCor gate。通过的 panel 才做二维条件密度的 branch 检测。


In [ ]:
(
    KDE2D_RESULTS_DF,
    KDE2D_RESULT_OBJECTS,
    KDE2D_GEOMETRY,
    _kde2d_elapsed,
) = _kde_branch_comparison.run_detector(
    BRANCH_KDE_CASES,
    method='kde2d',
    metrics=_BRANCH_ASSOCIATION_METRICS,
    mode=BRANCH_GATE_MODE,
    mic_min=BRANCH_MIC_MIN,
    dcor_min=BRANCH_DCOR_MIN,
    coverage_mode=BRANCH_COVERAGE_MODE,
    detect_behind_gate=DETECT_BEHIND_GATE,
)

print(
    f'Conditional 2-D KDE completed: {len(KDE2D_RESULTS_DF)} panels '
    f'in {_kde2d_elapsed:.1f}s'
)
display(_kde_branch_comparison.summarize(KDE2D_RESULTS_DF))

# --- Draw grids one pair at a time: scatter → KDE2D → next pair ---
for _, _, pair_label in VARIABLE_PAIRS:
    pair_slug = PAIR_SLUGS[pair_label]
    for version in ['raw', 'trimmed']:
        # Scatter grid (with KDE2D cross-comparison labels)
        scatter_path = _kde_branch_comparison.draw_scatter_grid(
            BRANCH_KDE_CASES,
            SCATTER_KDE_RESULT_OBJECTS,
            SCATTER_KDE_GEOMETRY,
            pair_label,
            version,
            FIGURE_DIR,
            pair_slug,
            show=False,
            other_objects=KDE2D_RESULT_OBJECTS,
            show_lowess=SHOW_LOWESS,
            lowess_turning_threshold=3,
        )
        print(f'Scatter KDE: {pair_label} \u00b7 {version.upper()}')
        display(_NotebookImage(filename=str(scatter_path)))

        # KDE2D grid (with scatter cross-comparison labels)
        kde2d_path = _kde_branch_comparison.draw_kde2d_grid(
            BRANCH_KDE_CASES,
            KDE2D_RESULT_OBJECTS,
            KDE2D_GEOMETRY,
            pair_label,
            version,
            FIGURE_DIR,
            pair_slug,
            show=False,
            other_objects=SCATTER_KDE_RESULT_OBJECTS,
        )
        print(f'2D KDE: {pair_label} \u00b7 {version.upper()}')
        display(_NotebookImage(filename=str(kde2d_path)))

if WRITE_OUTPUTS:
    SCATTER_KDE_RESULTS_DF.to_csv(
        S4_OUTPUT_DIR / 'S4_branch_scatter_window_kde_all_cases.csv',
        index=False,
    )
    KDE2D_RESULTS_DF.to_csv(
        S4_OUTPUT_DIR / 'S4_branch_conditional_2d_kde_all_cases.csv',
        index=False,
    )


## 16. Hexbin density — 5 × 6 grids / Hexbin 密度图

Same model × domain layout as Sections 14–15. Each panel is a log-count Hexbin of the original scatter, after the same MIC / dCor gate. Status labels are taken from the scatter-window KDE detector so the three 5×6 figures can be compared directly.

**中文说明：** 和第 14–15 节相同的 5 模型 × 6 分区布局。每个 panel 用 Hexbin（对数计数）看点的密度；MIC / dCor gate 和状态标注与散点窗口 KDE 一致，方便对照。


In [ ]:
import importlib

_kde_branch_comparison = importlib.reload(_kde_branch_comparison)

if 'SCATTER_KDE_RESULT_OBJECTS' not in globals():
    raise RuntimeError('Run Section 14 first so scatter-window KDE statuses exist.')
if 'KDE2D_RESULT_OBJECTS' not in globals():
    raise RuntimeError('Run Section 14 first so KDE2D result objects exist.')
if 'BRANCH_KDE_CASES' not in globals():
    raise RuntimeError('Run Section 14 first so BRANCH_KDE_CASES exists.')

for _, _, pair_label in VARIABLE_PAIRS:
    pair_slug = PAIR_SLUGS[pair_label]
    for version in ['raw', 'trimmed']:
        hexbin_path = _kde_branch_comparison.draw_hexbin_grid(
            BRANCH_KDE_CASES,
            SCATTER_KDE_RESULT_OBJECTS,
            pair_label,
            version,
            FIGURE_DIR,
            pair_slug,
            show=False,
            scatter_objects=SCATTER_KDE_RESULT_OBJECTS,
            kde2d_objects=KDE2D_RESULT_OBJECTS,
        )
        print(f'Hexbin: {pair_label} \u00b7 {version.upper()}')
        display(_NotebookImage(filename=str(hexbin_path)))
